
<div style="
    background-color:#4A1942;
    padding:25px;
    border-radius:15px;
    text-align:center;
    color:white;
    font-size:32px;
    font-weight:bold;
">
    02 — Retrieval Evaluation
</div>

<br>



## CardioPress AI
### Evaluating Retrieval Strategies for Cardiovascular RAG

# Retrieval Evaluation — Overview

## Objective

This notebook evaluates different information retrieval strategies for the
CardioPress AI RAG system using the validated cardiovascular-health dataset
prepared in the previous notebook.

The main goal is to determine:

1. Which chunking strategy provides better retrieval quality.
2. Which retrieval method performs best.
3. Which Top-K value provides the best evidence coverage.
4. Whether reranking improves retrieval performance.

---

## Input Artifacts

The validated artifacts generated in the previous notebook are:

- `chunks_A.json` — Experiment A chunks
- `chunks_B.json` — Experiment B chunks
- `metadata_pages.json` — page-level metadata and source traceability

These files are loaded directly without repeating PDF extraction,
cleaning, section detection, or chunking.

---

## Chunking Strategies

### Experiment A
- Target range: 400–600 tokens
- Overlap: 60 tokens
- Chunks: 73

### Experiment B
- Target range: 700–900 tokens
- Chunks: 43

Both strategies will be evaluated using the same retrieval benchmark.

---

## Retrieval Methods

The following retrieval approaches will be implemented and compared:

### 1. Semantic Retrieval
Dense vector embeddings will be used to retrieve chunks based on
semantic similarity between the query and document chunks.

### 2. BM25 Retrieval
A keyword-based retrieval approach will be implemented using BM25
to evaluate lexical matching.

### 3. Hybrid Retrieval
Semantic and BM25 retrieval results will be combined to improve
both semantic and keyword-based evidence retrieval.

### 4. Hybrid + Cross-Encoder Reranker
The hybrid candidate results will be reranked using a Cross-Encoder
to improve the final ordering of the most relevant evidence.

---

## Evaluation Benchmark

A fixed evaluation benchmark will be created and reused across all experiments.

The benchmark will contain different question types:

- Direct questions
- Paraphrased questions
- Abbreviation-based questions
- Threshold / numerical questions
- Out-of-scope questions

Each question will have expected evidence that can be matched against
the retrieved chunks.

---

## Evaluation Metrics

Retrieval performance will be measured using:

- Recall@3
- Recall@5
- Recall@10
- Mean Reciprocal Rank (MRR)

---

## Experimental Comparison

The evaluation will compare:

| Dimension | Values |
|---|---|
| Chunking | A vs B |
| Retrieval | Semantic vs BM25 vs Hybrid vs Reranked Hybrid |
| Top-K | 3, 5, 10 |
| Metrics | Recall@K, MRR |

The same benchmark and evaluation procedure will be used for every
configuration to ensure a fair comparison.

---

## Final Goal

The final result of this notebook will identify the best combination of:

**Chunking Strategy + Retrieval Method + Top-K**

for the CardioPress AI RAG system.

The selected configuration will be used in the final RAG pipeline.

In [1]:
# STEP 1 — Imports

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("Imports loaded successfully ✓")

Imports loaded successfully ✓


In [2]:
# STEP 2 — Load validated artifacts

DATA_DIR = Path("../Data/processed")

with open(DATA_DIR / "chunks_A.json", "r", encoding="utf-8") as f:
    chunks_A = json.load(f)

with open(DATA_DIR / "chunks_B.json", "r", encoding="utf-8") as f:
    chunks_B = json.load(f)

with open(DATA_DIR / "metadata_pages.json", "r", encoding="utf-8") as f:
    metadata_pages = json.load(f)

print("========== ARTIFACTS LOADED ==========")
print(f"Experiment A chunks : {len(chunks_A)}")
print(f"Experiment B chunks : {len(chunks_B)}")
print(f"Metadata records    : {len(metadata_pages)}")

========== ARTIFACTS LOADED ==========
Experiment A chunks : 73
Experiment B chunks : 43
Metadata records    : 178


In [3]:
# STEP 3 — Validate loaded artifacts

assert len(chunks_A) == 73, "Experiment A chunk count mismatch"
assert len(chunks_B) == 43, "Experiment B chunk count mismatch"
assert len(metadata_pages) == 178, "Metadata count mismatch"

print("========== ARTIFACT VALIDATION ==========")
print("Experiment A : 73 chunks ✓")
print("Experiment B : 43 chunks ✓")
print("Metadata     : 178 records ✓")
print("STEP 1 PASSED — Retrieval inputs validated ✓")

========== ARTIFACT VALIDATION ==========
Experiment A : 73 chunks ✓
Experiment B : 43 chunks ✓
Metadata     : 178 records ✓
STEP 1 PASSED — Retrieval inputs validated ✓


________________________________

______________________________

# STEP 2 — Embedding Setup

In [4]:
# STEP 2 — Load embedding model

from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Model:", MODEL_NAME)

embedding_model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

print("Embedding model loaded successfully ✓")

c:\Users\Rahma mohamed\OneDrive\Desktop\AI-Portfolio\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5420.70it/s]


Embedding model loaded successfully ✓


In [5]:
# STEP 2.1 — Validate embedding model

test_text = chunks_A[0]["text"]

test_embedding = embedding_model.encode(
    test_text,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("========== EMBEDDING VALIDATION ==========")
print("Embedding shape:", test_embedding.shape)
print("Embedding dtype:", test_embedding.dtype)

assert test_embedding.ndim == 1
assert len(test_embedding) == 384
assert np.isfinite(test_embedding).all()

print("Embedding dimension: 384 ✓")
print("Embedding values are valid ✓")
print("STEP 2 PASSED — Embedding model validated ✓")

========== EMBEDDING VALIDATION ==========
Embedding shape: (384,)
Embedding dtype: float32
Embedding dimension: 384 ✓
Embedding values are valid ✓
STEP 2 PASSED — Embedding model validated ✓


### STEP 3 — Create & Save Embeddings

In [6]:
# STEP 3 — Prepare texts for embedding

texts_A = [chunk["text"] for chunk in chunks_A]
texts_B = [chunk["text"] for chunk in chunks_B]

print("========== TEXT PREPARATION ==========")
print("Experiment A texts:", len(texts_A))
print("Experiment B texts:", len(texts_B))

assert len(texts_A) == 73
assert len(texts_B) == 43
assert all(isinstance(text, str) and text.strip() for text in texts_A)
assert all(isinstance(text, str) and text.strip() for text in texts_B)

print("All chunk texts are valid ✓")

========== TEXT PREPARATION ==========
Experiment A texts: 73
Experiment B texts: 43
All chunk texts are valid ✓


In [7]:
# STEP 3.1 — Generate embeddings

embeddings_A = embedding_model.encode(
    texts_A,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings_B = embedding_model.encode(
    texts_B,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\n========== EMBEDDINGS CREATED ==========")
print("Experiment A:", embeddings_A.shape)
print("Experiment B:", embeddings_B.shape)

Batches: 100%|██████████| 3/3 [00:00<00:00,  6.90it/s]


========== EMBEDDINGS CREATED ==========
Experiment A: (73, 384)
Experiment B: (43, 384)


In [8]:
# STEP 3.2 — Validate embeddings

assert embeddings_A.shape == (73, 384)
assert embeddings_B.shape == (43, 384)

assert np.isfinite(embeddings_A).all()
assert np.isfinite(embeddings_B).all()

# Because embeddings were normalized, their L2 norms should be ~1
norms_A = np.linalg.norm(embeddings_A, axis=1)
norms_B = np.linalg.norm(embeddings_B, axis=1)

assert np.allclose(norms_A, 1.0, atol=1e-5)
assert np.allclose(norms_B, 1.0, atol=1e-5)

print("========== EMBEDDING VALIDATION ==========")
print("A shape:", embeddings_A.shape, "✓")
print("B shape:", embeddings_B.shape, "✓")
print("A values finite ✓")
print("B values finite ✓")
print("Embeddings normalized ✓")
print("STEP 3 PASSED — Embeddings validated ✓")

========== EMBEDDING VALIDATION ==========
A shape: (73, 384) ✓
B shape: (43, 384) ✓
A values finite ✓
B values finite ✓
Embeddings normalized ✓
STEP 3 PASSED — Embeddings validated ✓


In [9]:
# STEP 3.3 — Save embeddings

import numpy as np

np.save(DATA_DIR / "embeddings_A.npy", embeddings_A)
np.save(DATA_DIR / "embeddings_B.npy", embeddings_B)

print("========== EMBEDDINGS SAVED ==========")
print("✓ embeddings_A.npy")
print("✓ embeddings_B.npy")
print("Saved to:", DATA_DIR.resolve())

========== EMBEDDINGS SAVED ==========
✓ embeddings_A.npy
✓ embeddings_B.npy
Saved to: C:\Users\Rahma mohamed\OneDrive\Desktop\AI-Portfolio\7__CardioPress AI\Data\processed


# STEP 2 — Embedding Preparation and Validation

## Objective
In this step, we convert the validated text chunks from both chunking experiments into numerical vector representations (embeddings) using the same embedding model:

**Model:** `sentence-transformers/all-MiniLM-L6-v2`

The model produces a **384-dimensional embedding vector** for each chunk.

## Experiments
- **Experiment A:** 73 chunks
- **Experiment B:** 43 chunks
- Each chunk is converted into one 384-dimensional vector.

## Validation
Before creating the embeddings, we validate that:
- The embedding model is loaded successfully.
- The embedding dimension is **384**.
- Input chunk texts are valid and non-empty.
- The generated embeddings contain only finite values.
- Embeddings are normalized to support cosine similarity retrieval.

## Output
The validated embeddings will be saved as:

- `embeddings_A.npy` → embeddings for Experiment A
- `embeddings_B.npy` → embeddings for Experiment B

These embeddings will be used in the next retrieval stage to compare the performance of the two chunking strategies.

___________________________________

______________________________________

# STEP 4 — Semantic Retrieval

In [10]:
# STEP 4 — Semantic Retrieval

from sklearn.metrics.pairwise import cosine_similarity


def semantic_search(
    query,
    chunks,
    embeddings,
    model,
    top_k=5
):
    """
    Retrieve the top-k most semantically similar chunks.
    """

    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).reshape(1, -1)

    scores = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "section_title": chunk["section_title"],
            "pages": chunk["pages"],
            "token_count": chunk["token_count"],
            "score": float(scores[idx]),
            "text": chunk["text"]
        })

    return results


print("Semantic retrieval function created ✓")

Semantic retrieval function created ✓


In [13]:
# STEP 4.1 — Semantic Retrieval

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query, chunks, embeddings, model, top_k=5):
    """
    Retrieve the most relevant chunks using semantic similarity.
    """

    # Encode query
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    # Make sure query embedding has correct shape
    query_embedding = np.asarray(query_embedding).reshape(1, -1)

    # Calculate cosine similarity
    scores = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    # Get top-K indices
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Build results
    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "section_title": chunk["section_title"],
            "page_numbers": chunk["page_numbers"],
            "token_count": chunk["token_count"],
            "score": float(scores[idx]),
            "text": chunk["text"],
            "source_url": chunk["source_url"]
        })

    return results

In [15]:
# STEP 4.0 — Load Embedding Model for Query Encoding

from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded successfully: ✓")
print("Model:", MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11441.94it/s]


Embedding model loaded successfully: ✓
Model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


C:\Users\Rahma mohamed\AppData\Local\Temp\ipykernel_26100\1667661297.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [16]:
# STEP 4.1 — Test Semantic Retrieval

test_query = "How does reducing salt intake help prevent hypertension?"

results_A = semantic_search(
    query=test_query,
    chunks=chunks_A,
    embeddings=embeddings_A,
    model=model,
    top_k=5
)

print("=" * 70)
print("SEMANTIC RETRIEVAL — EXPERIMENT A")
print("=" * 70)

for result in results_A:
    print(f"\nRank: {result['rank']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Section: {result['section_title']}")
    print(f"Pages: {result['page_numbers']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Tokens: {result['token_count']}")
    print(f"Text: {result['text'][:500]}...")

SEMANTIC RETRIEVAL — EXPERIMENT A

Rank: 1
Chunk ID: A_0047
Section: Module 3.2 — Hypertension
Pages: [254, 255, 256]
Score: 0.4649
Tokens: 600
Text: ##rice daily ( 5 ) tab. aspirin 150 mg once daily ( 6 ) tab. rosuvastatin 10 mg once daily step 3. ask each group to summarize and present the management plan. step 4. discuss common issues and challenges in the management of hypertension. facilitator ’ s explanatory note during the discussion, stimulate the participants to listen to the other groups with different case scenarios so that they can learn from each other. step 5. summarize and conclude the session with emphasis on evidence - based ...

Rank: 2
Chunk ID: A_0007
Section: Module 2.4 — Healthy Diet
Pages: [161, 162, 163]
Score: 0.4513
Tokens: 600
Text: pictures of unhealthy high fat foods ) eat this... ( picture of healthier options ) ¤ beware of salt ¢ minimize salt when eating or cooking. ¢ limit flavourings that contain a lot of salt, e. g. soy sauce, fish sauce, bouillon or 

# STEP 4 — Semantic Retrieval Evaluation

## Objective

In this step, we evaluate the semantic retrieval performance of the two validated chunking experiments:

- **Experiment A:** 400–600 tokens with overlap
- **Experiment B:** 700–900 tokens

Both experiments use the same embedding model:

`sentence-transformers/all-MiniLM-L6-v2`

with **384-dimensional normalized embeddings**.

## Initial Semantic Retrieval Test

A preliminary query was used to verify that semantic retrieval is working correctly:

> How does reducing salt intake help prevent hypertension?

The retrieved results showed relevant content from both:

- **Module 3.2 — Hypertension**
- **Module 2.4 — Healthy Diet**

This confirms that the semantic retrieval pipeline is functioning correctly and can retrieve related evidence across connected cardiovascular topics.

## Retrieval Evaluation Benchmark

We will now create a **fixed evaluation benchmark** so that all retrieval methods are evaluated fairly using the same questions and expected evidence.

The benchmark will include different question types:

1. **Direct questions**
   - Questions whose wording closely matches the source content.

2. **Paraphrased questions**
   - Questions asking for the same information using different wording.

3. **Abbreviation-based questions**
   - Questions using clinical abbreviations such as:
     - BP
     - SBP
     - DBP
     - CVD

4. **Threshold / numerical questions**
   - Questions requiring retrieval of specific clinical thresholds, values, doses, or recommendations.

5. **Out-of-scope questions**
   - Questions whose answers are not contained in the dataset.
   - These are used to test whether the retriever avoids returning misleading evidence.

Each benchmark question will have:
- Query
- Question type
- Expected evidence
- Expected section/module
- Expected page(s)
- Relevant chunk(s)

## Retrieval Methods to Compare

The same benchmark will be used to compare the following retrieval approaches:

### 1. Semantic Retrieval
Dense vector retrieval using cosine similarity between the query embedding and chunk embeddings.

### 2. BM25 Keyword Retrieval
Traditional lexical retrieval based on keyword matching and term importance.

### 3. Hybrid Retrieval
Combination of:
- Semantic similarity
- BM25 keyword relevance

The goal is to combine semantic understanding with exact clinical terminology and keyword matching.

### 4. Hybrid + Cross-Encoder Reranker
The Hybrid retrieval results will be further reranked using a Cross-Encoder to improve the ordering of the most relevant chunks.

## Top-K Evaluation

We will test multiple retrieval depths, especially:

- **Top-3**
- **Top-5**
- **Top-10**

This will allow us to determine how many retrieved chunks are needed to reliably include the required evidence.

## Evaluation Metrics

We will quantitatively evaluate all retrieval methods using:

### Recall@K

Measures whether the required evidence appears within the top K retrieved chunks.

We will calculate:

- Recall@3
- Recall@5
- Recall@10

### MRR — Mean Reciprocal Rank

Measures how highly the first relevant chunk is ranked.

A higher MRR means the relevant evidence appears closer to the top of the retrieval results.

## Final Comparison

The final evaluation will compare:

| Chunking | Retrieval Method | Top-K | Recall@K | MRR |
|---|---|---:|---:|---:|
| A | Semantic | 3/5/10 | — | — |
| B | Semantic | 3/5/10 | — | — |
| A | BM25 | 3/5/10 | — | — |
| B | BM25 | 3/5/10 | — | — |
| A | Hybrid | 3/5/10 | — | — |
| B | Hybrid | 3/5/10 | — | — |
| A | Hybrid + Reranker | 3/5/10 | — | — |
| B | Hybrid + Reranker | 3/5/10 | — | — |

The objective is to identify the best combination of:

**Chunking Strategy + Retrieval Method + Top-K**

for the final CardioPress AI RAG system.

_________________________________

____________________________________

# STEP 5 — Build the Fixed Retrieval Evaluation Benchmark


## Objective

In this step, we create a fixed evaluation benchmark for testing the retrieval system.

The same benchmark will be used for all retrieval approaches to ensure a fair and consistent comparison.

## Question Categories

The benchmark will contain five types of questions:

1. **Direct**
   - Questions that closely match information explicitly stated in the source.

2. **Paraphrased**
   - Questions that ask for the same information using different wording.

3. **Abbreviation**
   - Questions containing common clinical abbreviations such as BP, SBP, DBP, and CVD.

4. **Threshold / Numerical**
   - Questions requiring retrieval of specific numerical recommendations, thresholds, doses, or clinical targets.

5. **Out-of-Scope**
   - Questions whose answers are not available in the current dataset.
   - These questions are used to evaluate whether the retriever avoids confidently retrieving unrelated evidence.

## Expected Evidence

Each in-scope question will have manually defined expected evidence based on the original WHO source.

For each question we record:

- Query
- Question type
- Expected section
- Expected page(s)
- Expected evidence description

This allows us to calculate retrieval metrics objectively.

## Evaluation Consistency

The benchmark will remain fixed throughout the experiments.

The following retrieval methods will use exactly the same benchmark:

- Semantic Retrieval
- BM25
- Hybrid Retrieval
- Hybrid + Cross-Encoder Reranker

The benchmark will also be evaluated at:

- Top-3
- Top-5
- Top-10

## Evaluation Metrics

The benchmark will later be used to calculate:

- Recall@3
- Recall@5
- Recall@10
- MRR

The final results will be used to determine the best retrieval configuration for CardioPress AI.

# STEP 5.1 — Define Benchmark Questions

In [17]:
# STEP 5.1 — Define Fixed Evaluation Benchmark

evaluation_benchmark = [
    # ============================================================
    # 1. DIRECT QUESTIONS
    # ============================================================
    {
        "id": "Q01",
        "query": "How does reducing salt intake help prevent hypertension?",
        "question_type": "direct",
        "expected_section": "Module 2.4 — Healthy Diet",
        "expected_pages": [149, 150, 151, 152, 153],
        "expected_evidence": "Reducing salt intake to less than 5 g per day helps prevent hypertension and reduces cardiovascular risk."
    },

    {
        "id": "Q02",
        "query": "What is hypertension?",
        "question_type": "direct",
        "expected_section": "Module 3.2 — Hypertension",
        "expected_pages": [247],
        "expected_evidence": "Definition and basic description of hypertension."
    },

    {
        "id": "Q03",
        "query": "What are the recommended physical activity levels for adults?",
        "question_type": "direct",
        "expected_section": "Module 2.5 — Physical Activity",
        "expected_pages": [168],
        "expected_evidence": "Adults should perform at least 150 minutes of physical activity per week."
    },

    # ============================================================
    # 2. PARAPHRASED QUESTIONS
    # ============================================================
    {
        "id": "Q04",
        "query": "Why is eating less salt beneficial for people at risk of high blood pressure?",
        "question_type": "paraphrased",
        "expected_section": "Module 2.4 — Healthy Diet",
        "expected_pages": [149, 153],
        "expected_evidence": "Lower salt consumption helps prevent hypertension and reduces the risk of heart disease and stroke."
    },

    {
        "id": "Q05",
        "query": "What lifestyle activity can help control blood pressure and reduce cardiovascular risk?",
        "question_type": "paraphrased",
        "expected_section": "Module 2.5 — Physical Activity",
        "expected_pages": [168],
        "expected_evidence": "Regular physical activity can help control blood pressure and reduce the risk of hypertension, heart attack and stroke."
    },

    # ============================================================
    # 3. ABBREVIATION QUESTIONS
    # ============================================================
    {
        "id": "Q06",
        "query": "What lifestyle recommendations can help reduce BP?",
        "question_type": "abbreviation",
        "expected_section": "Module 3.2 — Hypertension",
        "expected_pages": [247, 254, 255, 256],
        "expected_evidence": "Lifestyle and management recommendations for hypertension and blood pressure control."
    },

    {
        "id": "Q07",
        "query": "What factors increase CVD risk?",
        "question_type": "abbreviation",
        "expected_section": "Module 3.1 — Cardiovascular Diseases",
        "expected_pages": [225],
        "expected_evidence": "Risk factors and prevention of cardiovascular diseases."
    },

    # ============================================================
    # 4. THRESHOLD / NUMERICAL QUESTIONS
    # ============================================================
    {
        "id": "Q08",
        "query": "How much salt should adults consume per day?",
        "question_type": "threshold",
        "expected_section": "Module 2.4 — Healthy Diet",
        "expected_pages": [149, 153],
        "expected_evidence": "Salt intake should be less than 5 g per day."
    },

    {
        "id": "Q09",
        "query": "How many minutes of physical activity should adults perform each week?",
        "question_type": "threshold",
        "expected_section": "Module 2.5 — Physical Activity",
        "expected_pages": [168],
        "expected_evidence": "Adults should perform at least 150 minutes of physical activity per week."
    },

    {
        "id": "Q10",
        "query": "What percentage of total energy intake should come from fat?",
        "question_type": "threshold",
        "expected_section": "Module 2.4 — Healthy Diet",
        "expected_pages": [149, 153],
        "expected_evidence": "Total fat should not exceed 30% of total energy intake."
    },

    # ============================================================
    # 5. OUT-OF-SCOPE QUESTIONS
    # ============================================================
    {
        "id": "Q11",
        "query": "What is the recommended treatment for bacterial pneumonia in children?",
        "question_type": "out_of_scope",
        "expected_section": None,
        "expected_pages": [],
        "expected_evidence": None
    },

    {
        "id": "Q12",
        "query": "What is the recommended insulin dose for type 1 diabetes?",
        "question_type": "out_of_scope",
        "expected_section": None,
        "expected_pages": [],
        "expected_evidence": None
    }
]

print("=" * 70)
print("BENCHMARK CREATED")
print("=" * 70)

print("Total questions:", len(evaluation_benchmark))

from collections import Counter

type_counts = Counter(
    q["question_type"]
    for q in evaluation_benchmark
)

print("\nQuestion distribution:")

for question_type, count in type_counts.items():
    print(f"- {question_type}: {count}")

BENCHMARK CREATED
Total questions: 12

Question distribution:
- direct: 3
- paraphrased: 2
- abbreviation: 2
- threshold: 3
- out_of_scope: 2


In [18]:
# STEP 5.2 — Validate Evaluation Benchmark

required_fields = [
    "id",
    "query",
    "question_type",
    "expected_section",
    "expected_pages",
    "expected_evidence"
]

errors = []

for question in evaluation_benchmark:

    # Check required fields
    for field in required_fields:
        if field not in question:
            errors.append(
                f"{question.get('id', 'UNKNOWN')}: missing {field}"
            )

    # Check query
    if not isinstance(question["query"], str) or not question["query"].strip():
        errors.append(
            f"{question['id']}: invalid query"
        )

    # Out-of-scope validation
    if question["question_type"] == "out_of_scope":
        if question["expected_pages"] != []:
            errors.append(
                f"{question['id']}: out-of-scope question should have no expected pages"
            )

# Check duplicate IDs
ids = [q["id"] for q in evaluation_benchmark]

if len(ids) != len(set(ids)):
    errors.append("Duplicate question IDs found")

print("=" * 70)
print("BENCHMARK VALIDATION")
print("=" * 70)

if errors:
    print("❌ Validation failed")
    for error in errors:
        print("-", error)
else:
    print("Total questions:", len(evaluation_benchmark))
    print("All required fields present: ✓")
    print("No duplicate IDs: ✓")
    print("Out-of-scope questions validated: ✓")
    print("✅ STEP 5.2 PASSED — BENCHMARK VALIDATED")

BENCHMARK VALIDATION
Total questions: 12
All required fields present: ✓
No duplicate IDs: ✓
Out-of-scope questions validated: ✓
✅ STEP 5.2 PASSED — BENCHMARK VALIDATED


# STEP 5.3 — Ground Truth Validation


## Objective

Before evaluating retrieval performance, the expected evidence for each benchmark question must be validated against the actual chunked dataset.

The purpose of this step is to ensure that:

- Expected sections exist in the dataset.
- Expected pages are represented by actual chunks.
- The expected evidence is actually present in the source text.
- Out-of-scope questions have no expected evidence.
- The benchmark provides reliable ground truth for Recall@K and MRR.

## Ground Truth Definition

For each in-scope question, a chunk will be considered relevant when it contains the expected evidence or directly represents the expected source information.

The validation will use:

- `chunk_id`
- `page_numbers`
- `section_title`
- `text`

The validated relevant chunk IDs will be stored with each benchmark question.

## Important

Ground truth will be based on the actual retrieved dataset rather than assuming that a module's starting page contains the answer.

This prevents incorrect evaluation caused by page-number assumptions.

After validation, the benchmark will be ready for quantitative retrieval evaluation.

In [19]:
# STEP 5.3 — Validate Ground Truth Against Actual Chunks

def find_candidate_chunks(question, chunks):
    """
    Find chunks that match the expected section and/or expected pages.
    """

    expected_section = question["expected_section"]
    expected_pages = set(question["expected_pages"])

    candidates = []

    for chunk in chunks:

        chunk_pages = set(chunk["page_numbers"])

        section_match = (
            expected_section is not None
            and chunk["section_title"] == expected_section
        )

        page_match = bool(
            expected_pages.intersection(chunk_pages)
        )

        if section_match or page_match:
            candidates.append(chunk)

    return candidates


print("=" * 70)
print("GROUND TRUTH CANDIDATE VALIDATION")
print("=" * 70)

for question in evaluation_benchmark:

    print(f"\n{question['id']} — {question['question_type']}")
    print("Query:", question["query"])

    # Out-of-scope
    if question["question_type"] == "out_of_scope":
        print("Expected evidence: NONE")
        print("Status: ✓ OUT-OF-SCOPE")
        continue

    candidates_A = find_candidate_chunks(
        question,
        chunks_A
    )

    candidates_B = find_candidate_chunks(
        question,
        chunks_B
    )

    print("Candidate chunks A:", len(candidates_A))
    print("Candidate chunks B:", len(candidates_B))

    if candidates_A:
        print(
            "A sample:",
            candidates_A[0]["chunk_id"],
            candidates_A[0]["page_numbers"]
        )

    if candidates_B:
        print(
            "B sample:",
            candidates_B[0]["chunk_id"],
            candidates_B[0]["page_numbers"]
        )

    if candidates_A and candidates_B:
        print("Status: ✓ Evidence represented in both experiments")

    else:
        print("Status: ⚠ Review required")

GROUND TRUTH CANDIDATE VALIDATION

Q01 — direct
Query: How does reducing salt intake help prevent hypertension?
Candidate chunks A: 8
Candidate chunks B: 5
A sample: A_0000 [149, 150, 151, 152, 153]
B sample: B_0000 [149, 150, 151, 152, 153]
Status: ✓ Evidence represented in both experiments

Q02 — direct
Query: What is hypertension?
Candidate chunks A: 10
Candidate chunks B: 6
A sample: A_0041 [245, 246, 247, 248, 249]
B sample: B_0024 [245, 246, 247, 248, 249]
Status: ✓ Evidence represented in both experiments

Q03 — direct
Query: What are the recommended physical activity levels for adults?
Candidate chunks A: 9
Candidate chunks B: 5
A sample: A_0008 [171, 172, 173, 174, 175]
B sample: B_0005 [171, 172, 173, 174, 175]
Status: ✓ Evidence represented in both experiments

Q04 — paraphrased
Query: Why is eating less salt beneficial for people at risk of high blood pressure?
Candidate chunks A: 8
Candidate chunks B: 5
A sample: A_0000 [149, 150, 151, 152, 153]
B sample: B_0000 [149, 150,

____________________________________________

__________________________________________

# STEP 6.1 — BM25 Keyword Retrieval



In this step, we implement BM25 keyword-based retrieval for both chunking experiments.

## Objective
Evaluate lexical retrieval based on exact and partial keyword matching, especially for:
- Direct questions
- Abbreviations
- Threshold-based questions
- Medical terminology

## Experiments
- Experiment A: 400–600 tokens with overlap
- Experiment B: 700–900 tokens

## Output
For each query, BM25 returns ranked candidate chunks with:
- Rank
- Chunk ID
- Section title
- Page numbers
- BM25 score
- Retrieved text

The same benchmark questions will be used later to compare BM25 with:
1. Semantic Retrieval
2. Hybrid Retrieval
3. Hybrid + Cross-Encoder Reranking

This ensures a fair comparison across all retrieval methods.

In [20]:
!pip install rank-bm25

In [21]:
# STEP 6.1.1 — Imports

import numpy as np
from rank_bm25 import BM25Okapi

print("BM25 imported successfully ✓")

BM25 imported successfully ✓


In [23]:
# STEP 6.1.2 — Prepare BM25 corpora

import re
import numpy as np
from rank_bm25 import BM25Okapi

def tokenize_for_bm25(text):
    return re.findall(r"\b\w+\b", text.lower())


corpus_A = [
    tokenize_for_bm25(chunk["text"])
    for chunk in chunks_A
]

corpus_B = [
    tokenize_for_bm25(chunk["text"])
    for chunk in chunks_B
]

bm25_A = BM25Okapi(corpus_A)
bm25_B = BM25Okapi(corpus_B)

print("========== BM25 CORPUS VALIDATION ==========")
print("Experiment A documents:", len(corpus_A))
print("Experiment B documents:", len(corpus_B))

assert len(corpus_A) == len(chunks_A)
assert len(corpus_B) == len(chunks_B)

print("A corpus ✓")
print("B corpus ✓")
print("BM25 A ✓")
print("BM25 B ✓")
print("STEP 6.1.2 PASSED — BM25 corpora validated ✓")

========== BM25 CORPUS VALIDATION ==========
Experiment A documents: 73
Experiment B documents: 43
A corpus ✓
B corpus ✓
BM25 A ✓
BM25 B ✓
STEP 6.1.2 PASSED — BM25 corpora validated ✓


In [24]:
# STEP 6.1.3 — BM25 Retrieval Function

def bm25_search(query, chunks, bm25, top_k=10):
    query_tokens = tokenize_for_bm25(query)

    scores = bm25.get_scores(query_tokens)

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(ranked_indices, start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "section_title": chunk["section_title"],
            "pages": chunk["page_numbers"],
            "token_count": chunk["token_count"],
            "score": float(scores[idx]),
            "text": chunk["text"]
        })

    return results

In [25]:
# STEP 6.1.4 — BM25 Retrieval Test

test_query = "How does reducing salt intake help prevent hypertension?"

results_A_bm25 = bm25_search(
    query=test_query,
    chunks=chunks_A,
    bm25=bm25_A,
    top_k=5
)

results_B_bm25 = bm25_search(
    query=test_query,
    chunks=chunks_B,
    bm25=bm25_B,
    top_k=5
)

print("=" * 70)
print("BM25 RETRIEVAL — EXPERIMENT A")
print("=" * 70)

for result in results_A_bm25:
    print(
        f"Rank: {result['rank']} | "
        f"Chunk: {result['chunk_id']} | "
        f"Pages: {result['pages']} | "
        f"Score: {result['score']:.4f}"
    )

print("\n" + "=" * 70)
print("BM25 RETRIEVAL — EXPERIMENT B")
print("=" * 70)

for result in results_B_bm25:
    print(
        f"Rank: {result['rank']} | "
        f"Chunk: {result['chunk_id']} | "
        f"Pages: {result['pages']} | "
        f"Score: {result['score']:.4f}"
    )

BM25 RETRIEVAL — EXPERIMENT A
Rank: 1 | Chunk: A_0007 | Pages: [161, 162, 163] | Score: 9.4914
Rank: 2 | Chunk: A_0005 | Pages: [158, 159, 160] | Score: 8.4926
Rank: 3 | Chunk: A_0071 | Pages: [320, 321] | Score: 7.3582
Rank: 4 | Chunk: A_0024 | Pages: [200, 201] | Score: 6.2730
Rank: 5 | Chunk: A_0067 | Pages: [316, 317] | Score: 5.3317

BM25 RETRIEVAL — EXPERIMENT B
Rank: 1 | Chunk: B_0004 | Pages: [160, 161, 162, 163] | Score: 7.4792
Rank: 2 | Chunk: B_0003 | Pages: [158, 159, 160] | Score: 6.6811
Rank: 3 | Chunk: B_0013 | Pages: [198, 199, 200] | Score: 5.4893
Rank: 4 | Chunk: B_0042 | Pages: [320, 321, 322] | Score: 5.2960
Rank: 5 | Chunk: B_0040 | Pages: [316, 317, 318, 319] | Score: 5.2904


In [26]:
# STEP 6.1.5 — Validate BM25 Retrieval

print("=" * 70)
print("BM25 RETRIEVAL VALIDATION")
print("=" * 70)

# Basic validation
assert len(results_A_bm25) == 5
assert len(results_B_bm25) == 5

# Required fields
required_fields = {
    "rank",
    "chunk_id",
    "section_title",
    "pages",
    "token_count",
    "score",
    "text"
}

for result in results_A_bm25:
    assert required_fields.issubset(result.keys())

for result in results_B_bm25:
    assert required_fields.issubset(result.keys())

# Rank validation
assert [r["rank"] for r in results_A_bm25] == [1, 2, 3, 4, 5]
assert [r["rank"] for r in results_B_bm25] == [1, 2, 3, 4, 5]

# Score validation
assert all(np.isfinite(r["score"]) for r in results_A_bm25)
assert all(np.isfinite(r["score"]) for r in results_B_bm25)

# Chunk IDs must be unique
assert len({r["chunk_id"] for r in results_A_bm25}) == 5
assert len({r["chunk_id"] for r in results_B_bm25}) == 5

print("Experiment A results: valid ✓")
print("Experiment B results: valid ✓")
print("Ranks: valid ✓")
print("Scores: finite ✓")
print("Chunk IDs: unique ✓")

print("\n" + "=" * 70)
print("STEP 6.1.5 PASSED — BM25 RETRIEVAL VALIDATED ✓")
print("=" * 70)

BM25 RETRIEVAL VALIDATION
Experiment A results: valid ✓
Experiment B results: valid ✓
Ranks: valid ✓
Scores: finite ✓
Chunk IDs: unique ✓

STEP 6.1.5 PASSED — BM25 RETRIEVAL VALIDATED ✓


In [33]:
# STEP 6.2.1 — Run BM25 on Full Benchmark

TOP_K = 10

bm25_results_A = {}
bm25_results_B = {}

for question in evaluation_benchmark:
    qid = question["id"]
    query = question["query"]

    bm25_results_A[qid] = bm25_search(
        query=query,
        chunks=chunks_A,
        bm25=bm25_A,
        top_k=TOP_K
    )

    bm25_results_B[qid] = bm25_search(
        query=query,
        chunks=chunks_B,
        bm25=bm25_B,
        top_k=TOP_K
    )

print("=" * 70)
print("BM25 FULL BENCHMARK")
print("=" * 70)

print("Questions evaluated:", len(evaluation_benchmark))
print("Experiment A:", len(bm25_results_A), "questions ✓")
print("Experiment B:", len(bm25_results_B), "questions ✓")

assert len(bm25_results_A) == 12
assert len(bm25_results_B) == 12

for qid in bm25_results_A:
    assert len(bm25_results_A[qid]) == TOP_K
    assert len(bm25_results_B[qid]) == TOP_K

print("Top-10 results available for all 12 questions ✓")
print("STEP 6.2.1 PASSED — FULL BM25 BENCHMARK ✓")

BM25 FULL BENCHMARK
Questions evaluated: 12
Experiment A: 12 questions ✓
Experiment B: 12 questions ✓
Top-10 results available for all 12 questions ✓
STEP 6.2.1 PASSED — FULL BM25 BENCHMARK ✓


In [34]:
# STEP 6.2.2 — Inspect BM25 Benchmark Results

sample_qid = evaluation_benchmark[0]["id"]

print("=" * 70)
print(f"BM25 RESULTS — {sample_qid}")
print("=" * 70)

print("\nQuery:")
print(evaluation_benchmark[0]["query"])

print("\nExpected section:")
print(evaluation_benchmark[0]["expected_section"])

print("\nExpected pages:")
print(evaluation_benchmark[0]["expected_pages"])

print("\n" + "-" * 70)
print("EXPERIMENT A")
print("-" * 70)

for r in bm25_results_A[sample_qid]:
    print(
        f"Rank {r['rank']} | "
        f"Chunk {r['chunk_id']} | "
        f"Pages {r['pages']} | "
        f"Score {r['score']:.4f}"
    )

print("\n" + "-" * 70)
print("EXPERIMENT B")
print("-" * 70)

for r in bm25_results_B[sample_qid]:
    print(
        f"Rank {r['rank']} | "
        f"Chunk {r['chunk_id']} | "
        f"Pages {r['pages']} | "
        f"Score {r['score']:.4f}"
    )

BM25 RESULTS — Q01

Query:
How does reducing salt intake help prevent hypertension?

Expected section:
Module 2.4 — Healthy Diet

Expected pages:
[149, 150, 151, 152, 153]

----------------------------------------------------------------------
EXPERIMENT A
----------------------------------------------------------------------
Rank 1 | Chunk A_0007 | Pages [161, 162, 163] | Score 9.4914
Rank 2 | Chunk A_0005 | Pages [158, 159, 160] | Score 8.4926
Rank 3 | Chunk A_0071 | Pages [320, 321] | Score 7.3582
Rank 4 | Chunk A_0024 | Pages [200, 201] | Score 6.2730
Rank 5 | Chunk A_0067 | Pages [316, 317] | Score 5.3317
Rank 6 | Chunk A_0023 | Pages [199, 200] | Score 5.0856
Rank 7 | Chunk A_0039 | Pages [236, 237] | Score 5.0178
Rank 8 | Chunk A_0066 | Pages [315, 316] | Score 4.2750
Rank 9 | Chunk A_0003 | Pages [155, 156, 157] | Score 4.2225
Rank 10 | Chunk A_0068 | Pages [317, 318, 319] | Score 4.1623

----------------------------------------------------------------------
EXPERIMENT B
------

In [35]:
# STEP 6.2.3 — Locate Ground Truth Candidate Data

variable_names = list(globals().keys())

print("=" * 70)
print("GROUND TRUTH / CANDIDATE VARIABLES")
print("=" * 70)

for name in variable_names:
    name_lower = name.lower()

    if (
        "candidate" in name_lower
        or "ground" in name_lower
        or "relevant" in name_lower
    ):
        try:
            value = globals()[name]
            print(
                f"{name} -> "
                f"{type(value).__name__} | "
                f"Length: {len(value) if hasattr(value, '__len__') else 'N/A'}"
            )
        except Exception:
            pass

GROUND TRUTH / CANDIDATE VARIABLES
find_candidate_chunks -> function | Length: N/A
candidates_A -> list | Length: 8
candidates_B -> list | Length: 5


___________________________________________

# STEP 6.3 — BM25 Evaluation: Recall@K and MRR



In this step, we evaluate the BM25 retrieval performance on the fixed
12-question evaluation benchmark.

The evaluation uses the validated ground-truth evidence for each question
and measures retrieval quality using:

- Recall@K: whether the expected evidence is retrieved within the top K results.
- MRR (Mean Reciprocal Rank): how high the first relevant result appears.

The evaluation is performed separately for:

- Experiment A: 400–600 tokens with overlap.
- Experiment B: 700–900 tokens.

The same benchmark and ground-truth definitions are used for both experiments
to ensure a fair comparison.

This step will provide the quantitative BM25 baseline that will later be
compared against Semantic Retrieval, Hybrid Retrieval, and Hybrid Retrieval
with Cross-Encoder Reranking.

In [37]:
# STEP 6.3.1 — BM25 Evaluation Functions

def is_relevant(result, expected_pages, expected_section=None):
    """
    A retrieved chunk is considered relevant if:
    1. It overlaps with the expected pages, OR
    2. It belongs to the expected section and overlaps with expected pages.
    """
    
    result_pages = set(result.get("pages", result.get("page_numbers", [])))
    expected_pages_set = set(expected_pages)

    page_overlap = bool(result_pages.intersection(expected_pages_set))

    if expected_section is not None:
        section_match = result.get("section_title") == expected_section
        return page_overlap and section_match

    return page_overlap


def calculate_recall_at_k(results, expected_pages, k):
    """
    Recall@K for retrieval:
    1 if at least one relevant chunk appears in top K,
    otherwise 0.
    """
    
    top_k = results[:k]

    for result in top_k:
        if is_relevant(result, expected_pages):
            return 1

    return 0


def calculate_reciprocal_rank(results, expected_pages):
    """
    Reciprocal Rank:
    1 / rank of the first relevant result.
    Returns 0 if no relevant result exists.
    """
    
    for rank, result in enumerate(results, start=1):
        if is_relevant(result, expected_pages):
            return 1.0 / rank

    return 0.0


print("=" * 70)
print("BM25 EVALUATION FUNCTIONS READY")
print("=" * 70)
print("Recall@K function: ✓")
print("MRR function: ✓")
print("Ground-truth matching: ✓")

BM25 EVALUATION FUNCTIONS READY
Recall@K function: ✓
MRR function: ✓
Ground-truth matching: ✓


In [38]:
# STEP 6.3.2 — Calculate BM25 Metrics

K_VALUES = [1, 3, 5, 10]

bm25_metrics_A = {}
bm25_metrics_B = {}

for experiment_name, bm25_results, metrics_dict in [
    ("A", bm25_results_A, bm25_metrics_A),
    ("B", bm25_results_B, bm25_metrics_B)
]:
    
    # Store per-question metrics
    per_question = {}

    for question in evaluation_benchmark:
        qid = question["id"]
        expected_pages = question["expected_pages"]

        results = bm25_results[qid]

        question_metrics = {}

        for k in K_VALUES:
            question_metrics[f"Recall@{k}"] = calculate_recall_at_k(
                results,
                expected_pages,
                k
            )

        question_metrics["RR"] = calculate_reciprocal_rank(
            results,
            expected_pages
        )

        per_question[qid] = question_metrics

    # Aggregate metrics
    metrics_dict["per_question"] = per_question

    for k in K_VALUES:
        metrics_dict[f"Recall@{k}"] = sum(
            per_question[qid][f"Recall@{k}"]
            for qid in per_question
        ) / len(per_question)

    metrics_dict["MRR"] = sum(
        per_question[qid]["RR"]
        for qid in per_question
    ) / len(per_question)


print("=" * 70)
print("BM25 METRICS CALCULATED")
print("=" * 70)

print("\nExperiment A")
for k in K_VALUES:
    print(f"Recall@{k}: {bm25_metrics_A[f'Recall@{k}']:.4f}")
print(f"MRR:       {bm25_metrics_A['MRR']:.4f}")

print("\nExperiment B")
for k in K_VALUES:
    print(f"Recall@{k}: {bm25_metrics_B[f'Recall@{k}']:.4f}")
print(f"MRR:       {bm25_metrics_B['MRR']:.4f}")

BM25 METRICS CALCULATED

Experiment A
Recall@1: 0.0000
Recall@3: 0.0000
Recall@5: 0.1667
Recall@10: 0.2500
MRR:       0.0452

Experiment B
Recall@1: 0.0000
Recall@3: 0.0833
Recall@5: 0.1667
Recall@10: 0.3333
MRR:       0.0671


In [39]:
# STEP 6.3.3 — BM25 Results Validation & Comparison

print("=" * 70)
print("BM25 RESULTS VALIDATION")
print("=" * 70)

# Validate metric ranges
for experiment_name, metrics in [
    ("A", bm25_metrics_A),
    ("B", bm25_metrics_B)
]:
    for k in K_VALUES:
        value = metrics[f"Recall@{k}"]
        assert 0.0 <= value <= 1.0, f"Invalid Recall@{k} for Experiment {experiment_name}"
    
    assert 0.0 <= metrics["MRR"] <= 1.0, \
        f"Invalid MRR for Experiment {experiment_name}"

print("Experiment A metrics: valid ✓")
print("Experiment B metrics: valid ✓")

# Compare A vs B
print("\n" + "=" * 70)
print("BM25 EXPERIMENT COMPARISON")
print("=" * 70)

print(f"{'Metric':<12} {'Experiment A':<18} {'Experiment B':<18} {'Winner'}")
print("-" * 70)

for k in K_VALUES:
    metric = f"Recall@{k}"
    a = bm25_metrics_A[metric]
    b = bm25_metrics_B[metric]
    
    if b > a:
        winner = "B"
    elif a > b:
        winner = "A"
    else:
        winner = "Tie"
    
    print(f"{metric:<12} {a:<18.4f} {b:<18.4f} {winner}")

a_mrr = bm25_metrics_A["MRR"]
b_mrr = bm25_metrics_B["MRR"]

if b_mrr > a_mrr:
    mrr_winner = "B"
elif a_mrr > b_mrr:
    mrr_winner = "A"
else:
    mrr_winner = "Tie"

print(f"{'MRR':<12} {a_mrr:<18.4f} {b_mrr:<18.4f} {mrr_winner}")

print("\n" + "=" * 70)
print("STEP 6.3.3 PASSED — BM25 RESULTS VALIDATED")
print("=" * 70)

BM25 RESULTS VALIDATION
Experiment A metrics: valid ✓
Experiment B metrics: valid ✓

BM25 EXPERIMENT COMPARISON
Metric       Experiment A       Experiment B       Winner
----------------------------------------------------------------------
Recall@1     0.0000             0.0000             Tie
Recall@3     0.0000             0.0833             B
Recall@5     0.1667             0.1667             Tie
Recall@10    0.2500             0.3333             B
MRR          0.0452             0.0671             B

STEP 6.3.3 PASSED — BM25 RESULTS VALIDATED


# STEP 6.4 — BM25 Per-Question Analysis

This step performs a question-level analysis of BM25 retrieval
performance across the fixed 12-question evaluation benchmark.

For each question, we inspect:

- The first relevant result rank.
- Reciprocal Rank (RR).
- Recall@1, Recall@3, Recall@5, and Recall@10.
- The difference between Experiment A and Experiment B.

The analysis also verifies the behavior of out-of-scope questions,
where no relevant evidence is expected.

This provides a detailed diagnostic view of the BM25 baseline before
moving to Hybrid Retrieval.

In [40]:
# STEP 6.4.1 — Generate Per-Question BM25 Analysis

def get_first_relevant_rank(results, expected_pages):
    """
    Return the rank of the first relevant result.
    Return None if no relevant result is found.
    """
    for rank, result in enumerate(results, start=1):
        if is_relevant(result, expected_pages):
            return rank
    return None


bm25_question_analysis = []

for question in evaluation_benchmark:
    qid = question["id"]
    query = question["query"]
    question_type = question["question_type"]
    expected_pages = question["expected_pages"]

    results_A = bm25_results_A[qid]
    results_B = bm25_results_B[qid]

    rr_A = calculate_reciprocal_rank(
        results_A,
        expected_pages
    )

    rr_B = calculate_reciprocal_rank(
        results_B,
        expected_pages
    )

    rank_A = get_first_relevant_rank(
        results_A,
        expected_pages
    )

    rank_B = get_first_relevant_rank(
        results_B,
        expected_pages
    )

    row = {
        "id": qid,
        "question_type": question_type,
        "query": query,

        "first_relevant_rank_A": rank_A,
        "first_relevant_rank_B": rank_B,

        "Recall@1_A": calculate_recall_at_k(
            results_A, expected_pages, 1
        ),
        "Recall@3_A": calculate_recall_at_k(
            results_A, expected_pages, 3
        ),
        "Recall@5_A": calculate_recall_at_k(
            results_A, expected_pages, 5
        ),
        "Recall@10_A": calculate_recall_at_k(
            results_A, expected_pages, 10
        ),

        "Recall@1_B": calculate_recall_at_k(
            results_B, expected_pages, 1
        ),
        "Recall@3_B": calculate_recall_at_k(
            results_B, expected_pages, 3
        ),
        "Recall@5_B": calculate_recall_at_k(
            results_B, expected_pages, 5
        ),
        "Recall@10_B": calculate_recall_at_k(
            results_B, expected_pages, 10
        ),

        "RR_A": rr_A,
        "RR_B": rr_B
    }

    bm25_question_analysis.append(row)


print("=" * 70)
print("BM25 PER-QUESTION ANALYSIS")
print("=" * 70)

print(f"Questions analyzed: {len(bm25_question_analysis)} ✓")

for row in bm25_question_analysis:
    print(
        f"{row['id']} | "
        f"Type: {row['question_type']} | "
        f"First relevant rank A: {row['first_relevant_rank_A']} | "
        f"First relevant rank B: {row['first_relevant_rank_B']} | "
        f"RR A: {row['RR_A']:.4f} | "
        f"RR B: {row['RR_B']:.4f}"
    )

BM25 PER-QUESTION ANALYSIS
Questions analyzed: 12 ✓
Q01 | Type: direct | First relevant rank A: None | First relevant rank B: None | RR A: 0.0000 | RR B: 0.0000
Q02 | Type: direct | First relevant rank A: 5 | First relevant rank B: 4 | RR A: 0.2000 | RR B: 0.2500
Q03 | Type: direct | First relevant rank A: None | First relevant rank B: None | RR A: 0.0000 | RR B: 0.0000
Q04 | Type: paraphrased | First relevant rank A: None | First relevant rank B: None | RR A: 0.0000 | RR B: 0.0000
Q05 | Type: paraphrased | First relevant rank A: None | First relevant rank B: None | RR A: 0.0000 | RR B: 0.0000
Q06 | Type: abbreviation | First relevant rank A: None | First relevant rank B: 9 | RR A: 0.0000 | RR B: 0.1111
Q07 | Type: abbreviation | First relevant rank A: 5 | First relevant rank B: 3 | RR A: 0.2000 | RR B: 0.3333
Q08 | Type: threshold | First relevant rank A: None | First relevant rank B: None | RR A: 0.0000 | RR B: 0.0000
Q09 | Type: threshold | First relevant rank A: None | First releva

# STEP 6.4.2 — BM25 Out-of-Scope Validation

This step validates BM25 behavior for out-of-scope questions.

For out-of-scope questions, the benchmark defines the expected evidence
as NONE. Therefore, these questions must not be treated as normal
retrieval-success cases.

The purpose of this step is to verify that:

- Out-of-scope questions are correctly identified in the benchmark.
- No ground-truth evidence is expected for these questions.
- Their zero Recall and zero Reciprocal Rank are interpreted correctly.
- They are excluded from the positive retrieval-performance calculation
  when analyzing in-scope retrieval quality.

This preserves a fair evaluation of BM25 while keeping the
out-of-scope safety behavior separately measurable.

In [42]:
# STEP 6.4.2 — Out-of-Scope Validation (Schema-Safe)

out_of_scope_questions = [
    q for q in evaluation_benchmark
    if q["question_type"] == "out_of_scope"
]

print("=" * 70)
print("BM25 OUT-OF-SCOPE VALIDATION")
print("=" * 70)

print(f"Out-of-scope questions: {len(out_of_scope_questions)}")

assert len(out_of_scope_questions) > 0, \
    "No out-of-scope questions found in benchmark."

for question in out_of_scope_questions:
    qid = question["id"]

    print(f"\n{qid}")
    print(f"Query: {question['query']}")
    print(f"Expected evidence: {question['expected_evidence']}")

    # Out-of-scope status is defined by the benchmark question type.
    assert question["question_type"] == "out_of_scope", \
        f"{qid} is not marked as out_of_scope."

    print("Question type validated: ✓")

print("\n" + "=" * 70)
print("OUT-OF-SCOPE QUESTIONS VALIDATED")
print("=" * 70)
print("Benchmark classification: ✓")
print("Ground-truth handling: ✓")
print("STEP 6.4.2 PASSED")
print("=" * 70)

BM25 OUT-OF-SCOPE VALIDATION
Out-of-scope questions: 2

Q11
Query: What is the recommended treatment for bacterial pneumonia in children?
Expected evidence: None
Question type validated: ✓

Q12
Query: What is the recommended insulin dose for type 1 diabetes?
Expected evidence: None
Question type validated: ✓

OUT-OF-SCOPE QUESTIONS VALIDATED
Benchmark classification: ✓
Ground-truth handling: ✓
STEP 6.4.2 PASSED


# STEP 6.4.3 — BM25 Performance by Question Type

This step analyzes BM25 retrieval performance according to question type.

The benchmark contains five question categories:

- Direct
- Paraphrased
- Abbreviation
- Threshold
- Out-of-scope

For each in-scope category, we calculate:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR

Out-of-scope questions are reported separately and are not included
in positive retrieval-performance metrics.

This analysis helps identify the types of clinical questions for which
keyword-based retrieval performs well or poorly.

In [43]:
# STEP 6.4.3 — BM25 Performance by Question Type

from collections import defaultdict

IN_SCOPE_TYPES = [
    "direct",
    "paraphrased",
    "abbreviation",
    "threshold"
]

question_type_metrics_A = defaultdict(list)
question_type_metrics_B = defaultdict(list)

for row in bm25_question_analysis:

    qtype = row["question_type"]

    if qtype not in IN_SCOPE_TYPES:
        continue

    for k in K_VALUES:
        question_type_metrics_A[qtype].append(
            row[f"Recall@{k}_A"]
        )
        question_type_metrics_B[qtype].append(
            row[f"Recall@{k}_B"]
        )


print("=" * 90)
print("BM25 PERFORMANCE BY QUESTION TYPE")
print("=" * 90)

print(
    f"{'Question Type':<15}"
    f"{'R@1 A':<10}{'R@1 B':<10}"
    f"{'R@3 A':<10}{'R@3 B':<10}"
    f"{'R@5 A':<10}{'R@5 B':<10}"
    f"{'R@10 A':<10}{'R@10 B':<10}"
)

print("-" * 90)

for qtype in IN_SCOPE_TYPES:

    rows = [
        row for row in bm25_question_analysis
        if row["question_type"] == qtype
    ]

    if not rows:
        continue

    values = {}

    for k in K_VALUES:
        values[f"R@{k}_A"] = sum(
            row[f"Recall@{k}_A"] for row in rows
        ) / len(rows)

        values[f"R@{k}_B"] = sum(
            row[f"Recall@{k}_B"] for row in rows
        ) / len(rows)

    print(
        f"{qtype:<15}"
        f"{values['R@1_A']:<10.4f}{values['R@1_B']:<10.4f}"
        f"{values['R@3_A']:<10.4f}{values['R@3_B']:<10.4f}"
        f"{values['R@5_A']:<10.4f}{values['R@5_B']:<10.4f}"
        f"{values['R@10_A']:<10.4f}{values['R@10_B']:<10.4f}"
    )


# MRR by question type
print("\n" + "=" * 70)
print("MRR BY QUESTION TYPE")
print("=" * 70)

print(f"{'Question Type':<15}{'MRR A':<15}{'MRR B':<15}")

print("-" * 70)

for qtype in IN_SCOPE_TYPES:

    rows = [
        row for row in bm25_question_analysis
        if row["question_type"] == qtype
    ]

    if not rows:
        continue

    mrr_A = sum(row["RR_A"] for row in rows) / len(rows)
    mrr_B = sum(row["RR_B"] for row in rows) / len(rows)

    print(
        f"{qtype:<15}"
        f"{mrr_A:<15.4f}"
        f"{mrr_B:<15.4f}"
    )


print("\n" + "=" * 90)
print("STEP 6.4.3 COMPLETED — QUESTION TYPE ANALYSIS")
print("=" * 90)

BM25 PERFORMANCE BY QUESTION TYPE
Question Type  R@1 A     R@1 B     R@3 A     R@3 B     R@5 A     R@5 B     R@10 A    R@10 B    
------------------------------------------------------------------------------------------
direct         0.0000    0.0000    0.0000    0.0000    0.3333    0.3333    0.3333    0.3333    
paraphrased    0.0000    0.0000    0.0000    0.0000    0.0000    0.0000    0.0000    0.0000    
abbreviation   0.0000    0.0000    0.0000    0.5000    0.5000    0.5000    0.5000    1.0000    
threshold      0.0000    0.0000    0.0000    0.0000    0.0000    0.0000    0.3333    0.3333    

MRR BY QUESTION TYPE
Question Type  MRR A          MRR B          
----------------------------------------------------------------------
direct         0.0667         0.0833         
paraphrased    0.0000         0.0000         
abbreviation   0.1000         0.2222         
threshold      0.0476         0.0370         

STEP 6.4.3 COMPLETED — QUESTION TYPE ANALYSIS


# STEP 6.4.4 — BM25 Diagnostic Summary

The BM25 evaluation shows that keyword-based retrieval has limited
performance on the clinical question benchmark.

Key observations:

- Overall Experiment B performs better than Experiment A in BM25,
  particularly for Recall@3, Recall@10, and MRR.
- Direct questions achieve moderate retrieval performance at higher K,
  but Recall@1 and Recall@3 remain limited.
- Paraphrased questions are not successfully retrieved by BM25 in either
  chunking experiment, demonstrating the limitation of lexical matching
  when the query wording differs from the source text.
- Abbreviation questions benefit substantially from the larger
  700–900-token chunks, with Experiment B achieving higher Recall@3
  and Recall@10.
- Threshold questions remain difficult for both experiments and show
  relatively low MRR.
- Out-of-scope questions are handled separately and are not treated as
  positive retrieval cases.

Therefore, BM25 provides a useful lexical retrieval baseline but is not
sufficient as the final retrieval strategy for this clinical RAG system.

The next stage will combine semantic similarity with BM25 using Hybrid
Retrieval to improve robustness to paraphrasing, terminology variation,
abbreviations, and clinically relevant threshold queries.

_________________________________________

_______________________________________

# STEP 7 — Hybrid Retrieval



This stage combines lexical and semantic retrieval to improve the
robustness of the clinical RAG retrieval pipeline.

The hybrid approach combines:

- BM25 retrieval for exact clinical terms, keywords, abbreviations,
  numbers, and threshold values.
- Semantic retrieval using sentence embeddings for paraphrased and
  semantically related queries.

Two chunking experiments will be evaluated:

- Experiment A: 400–600 tokens with overlap.
- Experiment B: 700–900 tokens.

The same fixed 12-question benchmark and validated ground-truth evidence
used in the previous experiments will be reused.

The evaluation will compare different hybrid weighting strategies and
measure:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR

The goal is to determine whether combining lexical and semantic signals
improves retrieval over BM25 and semantic retrieval individually.

No changes will be made to the benchmark, ground truth, chunking,
embeddings, or source metadata during this stage.

In [44]:
# STEP 7.1 — Check Hybrid Retrieval Inputs

required_variables = [
    "evaluation_benchmark",
    "bm25_results_A",
    "bm25_results_B",
    "chunks_A",
    "chunks_B",
    "embeddings_A",
    "embeddings_B"
]

print("=" * 70)
print("HYBRID RETRIEVAL INPUT VALIDATION")
print("=" * 70)

for variable in required_variables:
    exists = variable in globals()
    print(f"{variable}: {'✓' if exists else '✗'}")

    if not exists:
        raise NameError(
            f"Required variable '{variable}' is not available."
        )

print("\nBenchmark questions:", len(evaluation_benchmark))
print("BM25 A questions:", len(bm25_results_A))
print("BM25 B questions:", len(bm25_results_B))

assert len(evaluation_benchmark) == 12
assert len(bm25_results_A) == 12
assert len(bm25_results_B) == 12

print("\nSTEP 7.1 PASSED — HYBRID INPUTS VALIDATED ✓")

HYBRID RETRIEVAL INPUT VALIDATION
evaluation_benchmark: ✓
bm25_results_A: ✓
bm25_results_B: ✓
chunks_A: ✓
chunks_B: ✓
embeddings_A: ✓
embeddings_B: ✓

Benchmark questions: 12
BM25 A questions: 12
BM25 B questions: 12

STEP 7.1 PASSED — HYBRID INPUTS VALIDATED ✓


# STEP 7.2 — Semantic Retrieval for Full Benchmark

Before constructing the hybrid retrieval system, semantic retrieval
results are generated for all questions in the fixed evaluation
benchmark.

The same semantic retrieval pipeline validated earlier is reused for:

- Experiment A: 400–600 tokens with overlap.
- Experiment B: 700–900 tokens.

The benchmark and ground-truth definitions remain unchanged.

The resulting Top-10 semantic rankings will be combined later with
BM25 rankings to construct the hybrid retrieval scores.

This step ensures that both retrieval signals are available for every
benchmark question before hybrid scoring.

In [45]:
# STEP 7.2 — Generate Semantic Results for Full Benchmark

semantic_results_A = {}
semantic_results_B = {}

for question in evaluation_benchmark:

    qid = question["id"]
    query = question["query"]

    semantic_results_A[qid] = semantic_search(
        query=query,
        chunks=chunks_A,
        embeddings=embeddings_A,
        model=model,
        top_k=10
    )

    semantic_results_B[qid] = semantic_search(
        query=query,
        chunks=chunks_B,
        embeddings=embeddings_B,
        model=model,
        top_k=10
    )


print("=" * 70)
print("SEMANTIC FULL BENCHMARK")
print("=" * 70)

print(f"Questions evaluated: {len(evaluation_benchmark)}")
print(f"Experiment A: {len(semantic_results_A)} questions ✓")
print(f"Experiment B: {len(semantic_results_B)} questions ✓")

for qid in semantic_results_A:
    assert len(semantic_results_A[qid]) == 10
    assert len(semantic_results_B[qid]) == 10

print("Top-10 results available for all questions ✓")
print("STEP 7.2 PASSED — SEMANTIC RESULTS READY ✓")

SEMANTIC FULL BENCHMARK
Questions evaluated: 12
Experiment A: 12 questions ✓
Experiment B: 12 questions ✓
Top-10 results available for all questions ✓
STEP 7.2 PASSED — SEMANTIC RESULTS READY ✓


# STEP 7.3 — Normalize BM25 and Semantic Scores

BM25 and semantic retrieval produce scores with different scales and
interpretations.

Therefore, their raw scores cannot be directly added together.

In this step, retrieval scores are normalized independently within each
query and experiment using Min-Max normalization:

    normalized_score = (score - min_score) / (max_score - min_score)

This converts both retrieval signals to a comparable range from 0 to 1.

The normalization is performed separately for:

- BM25 Experiment A
- Semantic Experiment A
- BM25 Experiment B
- Semantic Experiment B

The normalized scores will be used in the next step to construct the
hybrid retrieval ranking.

No benchmark, chunk, metadata, or ground-truth information is modified.

In [46]:
# STEP 7.3.1 — Score Normalization

def min_max_normalize(scores):
    """
    Normalize scores to [0, 1] using Min-Max normalization.
    """
    scores = [float(score) for score in scores]

    if not scores:
        return []

    min_score = min(scores)
    max_score = max(scores)

    # If all scores are identical, assign 1.0 to all results.
    if max_score == min_score:
        return [1.0] * len(scores)

    return [
        (score - min_score) / (max_score - min_score)
        for score in scores
    ]


print("=" * 70)
print("SCORE NORMALIZATION FUNCTION")
print("=" * 70)

test_scores = [1.0, 2.0, 4.0, 8.0]
test_normalized = min_max_normalize(test_scores)

print("Original scores:", test_scores)
print("Normalized:", test_normalized)

assert min(test_normalized) == 0.0
assert max(test_normalized) == 1.0

print("Normalization range: [0, 1] ✓")
print("STEP 7.3.1 PASSED ✓")

SCORE NORMALIZATION FUNCTION
Original scores: [1.0, 2.0, 4.0, 8.0]
Normalized: [0.0, 0.14285714285714285, 0.42857142857142855, 1.0]
Normalization range: [0, 1] ✓
STEP 7.3.1 PASSED ✓


### STEP 7.3.2 — Normalize Scores for the Full Benchmark

For each benchmark question, BM25 and semantic retrieval scores are
normalized independently using Min-Max normalization.

Normalization is performed separately for Experiment A and Experiment B.

Each retrieval result receives a normalized score in the range [0, 1].
These normalized scores provide a common scale for combining the lexical
and semantic retrieval signals.

The original retrieval scores are preserved and are not modified.

In [47]:
# STEP 7.3.2 — Normalize Full Benchmark Scores

def normalize_results(results):
    """
    Add a normalized_score field while preserving the original score.
    """
    if not results:
        return []

    scores = [float(result["score"]) for result in results]
    normalized_scores = min_max_normalize(scores)

    normalized_results = []

    for result, normalized_score in zip(results, normalized_scores):
        item = result.copy()
        item["normalized_score"] = float(normalized_score)
        normalized_results.append(item)

    return normalized_results


# --------------------------------------------------
# Normalize all BM25 and Semantic results
# --------------------------------------------------

normalized_bm25_A = {}
normalized_bm25_B = {}

normalized_semantic_A = {}
normalized_semantic_B = {}


for question in evaluation_benchmark:

    qid = question["id"]

    normalized_bm25_A[qid] = normalize_results(
        bm25_results_A[qid]
    )

    normalized_bm25_B[qid] = normalize_results(
        bm25_results_B[qid]
    )

    normalized_semantic_A[qid] = normalize_results(
        semantic_results_A[qid]
    )

    normalized_semantic_B[qid] = normalize_results(
        semantic_results_B[qid]
    )


# --------------------------------------------------
# Validation
# --------------------------------------------------

print("=" * 70)
print("NORMALIZED RETRIEVAL SCORES")
print("=" * 70)

print(
    f"BM25 A: {len(normalized_bm25_A)} questions ✓"
)

print(
    f"BM25 B: {len(normalized_bm25_B)} questions ✓"
)

print(
    f"Semantic A: {len(normalized_semantic_A)} questions ✓"
)

print(
    f"Semantic B: {len(normalized_semantic_B)} questions ✓"
)


for qid in normalized_bm25_A:

    assert len(normalized_bm25_A[qid]) == 10
    assert len(normalized_bm25_B[qid]) == 10

    assert len(normalized_semantic_A[qid]) == 10
    assert len(normalized_semantic_B[qid]) == 10

    for result in (
        normalized_bm25_A[qid]
        + normalized_bm25_B[qid]
        + normalized_semantic_A[qid]
        + normalized_semantic_B[qid]
    ):
        assert 0.0 <= result["normalized_score"] <= 1.0
        assert "score" in result


print("\nAll normalized scores are within [0, 1] ✓")
print("Original scores preserved ✓")
print("Top-10 results preserved ✓")

print("\nSTEP 7.3.2 PASSED — SCORES NORMALIZED ✓")

NORMALIZED RETRIEVAL SCORES
BM25 A: 12 questions ✓
BM25 B: 12 questions ✓
Semantic A: 12 questions ✓
Semantic B: 12 questions ✓

All normalized scores are within [0, 1] ✓
Original scores preserved ✓
Top-10 results preserved ✓

STEP 7.3.2 PASSED — SCORES NORMALIZED ✓


# STEP 7.4 — Hybrid Score Construction

Hybrid retrieval combines semantic similarity and lexical BM25 evidence
into a single ranking score.

The hybrid score is defined as:

    HybridScore = α × SemanticScore + (1 − α) × BM25Score

where:

- α controls the contribution of semantic retrieval.
- (1 − α) controls the contribution of BM25 retrieval.

Before combination, both scores have already been normalized to [0, 1].

The same chunk may appear in both retrieval systems. When this happens,
its semantic and BM25 scores are combined for the same chunk.

Chunks appearing in only one retrieval system receive a score of zero
from the missing retrieval signal.

Multiple α values will be evaluated to determine the best balance
between semantic and lexical retrieval.

The benchmark and ground-truth definitions remain unchanged.

In [48]:
# STEP 7.4.1 — Build Hybrid Rankings

def build_hybrid_results(
    bm25_results,
    semantic_results,
    alpha=0.5,
    top_k=10
):
    """
    Combine normalized BM25 and semantic scores.

    HybridScore =
        alpha * SemanticScore
        + (1 - alpha) * BM25Score
    """

    combined = {}

    # -----------------------------
    # Add BM25 results
    # -----------------------------
    for result in bm25_results:

        chunk_id = result["chunk_id"]

        combined[chunk_id] = {
            "chunk_id": chunk_id,
            "bm25_score": float(
                result.get("normalized_score", 0.0)
            ),
            "semantic_score": 0.0,
            "section_title": result["section_title"],
            "pages": result["pages"],
            "token_count": result["token_count"],
            "text": result["text"]
        }

    # -----------------------------
    # Add semantic results
    # -----------------------------
    for result in semantic_results:

        chunk_id = result["chunk_id"]

        if chunk_id not in combined:

            combined[chunk_id] = {
                "chunk_id": chunk_id,
                "bm25_score": 0.0,
                "semantic_score": float(
                    result.get("normalized_score", 0.0)
                ),
                "section_title": result["section_title"],
                "pages": result["pages"],
                "token_count": result["token_count"],
                "text": result["text"]
            }

        else:

            combined[chunk_id]["semantic_score"] = float(
                result.get("normalized_score", 0.0)
            )

    # -----------------------------
    # Calculate hybrid score
    # -----------------------------
    hybrid_results = []

    for item in combined.values():

        hybrid_score = (
            alpha * item["semantic_score"]
            + (1 - alpha) * item["bm25_score"]
        )

        item["hybrid_score"] = float(hybrid_score)

        hybrid_results.append(item)

    # -----------------------------
    # Sort by hybrid score
    # -----------------------------
    hybrid_results.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    # -----------------------------
    # Add rank
    # -----------------------------
    hybrid_results = hybrid_results[:top_k]

    for rank, result in enumerate(
        hybrid_results,
        start=1
    ):
        result["rank"] = rank

    return hybrid_results


print("=" * 70)
print("HYBRID SCORE FUNCTION")
print("=" * 70)

print("Formula:")
print("HybridScore = α × Semantic + (1 − α) × BM25")
print("Same chunks: signals combined ✓")
print("Missing signal: score = 0 ✓")
print("Ranking: descending HybridScore ✓")

print("\nSTEP 7.4.1 READY ✓")

HYBRID SCORE FUNCTION
Formula:
HybridScore = α × Semantic + (1 − α) × BM25
Same chunks: signals combined ✓
Missing signal: score = 0 ✓
Ranking: descending HybridScore ✓

STEP 7.4.1 READY ✓


## STEP 7.4.2 — Hybrid Retrieval with α = 0.5

The first hybrid retrieval experiment uses an equal contribution from
semantic and lexical retrieval:

    HybridScore = 0.5 × SemanticScore + 0.5 × BM25Score

The experiment is performed independently for:

- Experiment A: 400–600 token chunks.
- Experiment B: 700–900 token chunks.

For each benchmark question, the Top-10 hybrid results are generated.

This equal-weight configuration provides a neutral baseline before
testing alternative semantic/BM25 weighting strategies.

In [50]:
# STEP 7.4.1 — Fix Hybrid Metadata Compatibility

def get_result_pages(result):
    """
    Safely retrieve page information from retrieval results.
    Supports both 'pages' and 'page_numbers'.
    """

    if "pages" in result:
        return result["pages"]

    if "page_numbers" in result:
        return result["page_numbers"]

    if "page_number" in result:
        return [result["page_number"]]

    return []


def build_hybrid_results(
    bm25_results,
    semantic_results,
    alpha=0.5,
    top_k=10
):
    """
    Combine normalized BM25 and semantic scores.

    HybridScore =
        alpha * SemanticScore
        + (1 - alpha) * BM25Score
    """

    combined = {}

    # ==================================================
    # 1. Add BM25 results
    # ==================================================

    for result in bm25_results:

        chunk_id = result["chunk_id"]

        combined[chunk_id] = {
            "chunk_id": chunk_id,
            "bm25_score": float(
                result.get("normalized_score", 0.0)
            ),
            "semantic_score": 0.0,
            "section_title": result["section_title"],
            "pages": get_result_pages(result),
            "token_count": result["token_count"],
            "text": result["text"]
        }

    # ==================================================
    # 2. Add Semantic results
    # ==================================================

    for result in semantic_results:

        chunk_id = result["chunk_id"]

        semantic_score = float(
            result.get("normalized_score", 0.0)
        )

        if chunk_id not in combined:

            combined[chunk_id] = {
                "chunk_id": chunk_id,
                "bm25_score": 0.0,
                "semantic_score": semantic_score,
                "section_title": result["section_title"],
                "pages": get_result_pages(result),
                "token_count": result["token_count"],
                "text": result["text"]
            }

        else:

            combined[chunk_id]["semantic_score"] = semantic_score

    # ==================================================
    # 3. Calculate Hybrid Score
    # ==================================================

    hybrid_results = []

    for item in combined.values():

        hybrid_score = (
            alpha * item["semantic_score"]
            + (1.0 - alpha) * item["bm25_score"]
        )

        item["hybrid_score"] = float(hybrid_score)

        hybrid_results.append(item)

    # ==================================================
    # 4. Sort by Hybrid Score
    # ==================================================

    hybrid_results.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    # ==================================================
    # 5. Keep Top-K
    # ==================================================

    hybrid_results = hybrid_results[:top_k]

    # ==================================================
    # 6. Assign Ranks
    # ==================================================

    for rank, result in enumerate(
        hybrid_results,
        start=1
    ):
        result["rank"] = rank

    return hybrid_results


print("=" * 70)
print("HYBRID SCORE FUNCTION — FIXED")
print("=" * 70)

print("Supports 'pages': ✓")
print("Supports 'page_numbers': ✓")
print("Supports 'page_number': ✓")
print("Same chunks combined: ✓")
print("Missing retrieval signal = 0: ✓")
print("Hybrid ranking: ✓")

print("\nSTEP 7.4.1 PASSED — HYBRID FUNCTION FIXED ✓")

HYBRID SCORE FUNCTION — FIXED
Supports 'pages': ✓
Supports 'page_numbers': ✓
Supports 'page_number': ✓
Same chunks combined: ✓
Missing retrieval signal = 0: ✓
Hybrid ranking: ✓

STEP 7.4.1 PASSED — HYBRID FUNCTION FIXED ✓


In [51]:
# STEP 7.4.2 — Hybrid Retrieval (α = 0.5)

ALPHA_BASELINE = 0.5

hybrid_results_A_05 = {}
hybrid_results_B_05 = {}

for question in evaluation_benchmark:

    qid = question["id"]

    hybrid_results_A_05[qid] = build_hybrid_results(
        bm25_results=normalized_bm25_A[qid],
        semantic_results=normalized_semantic_A[qid],
        alpha=ALPHA_BASELINE,
        top_k=10
    )

    hybrid_results_B_05[qid] = build_hybrid_results(
        bm25_results=normalized_bm25_B[qid],
        semantic_results=normalized_semantic_B[qid],
        alpha=ALPHA_BASELINE,
        top_k=10
    )


print("=" * 70)
print("HYBRID RETRIEVAL — α = 0.5")
print("=" * 70)

print(f"Experiment A: {len(hybrid_results_A_05)} questions ✓")
print(f"Experiment B: {len(hybrid_results_B_05)} questions ✓")

for qid in hybrid_results_A_05:

    assert len(hybrid_results_A_05[qid]) == 10
    assert len(hybrid_results_B_05[qid]) == 10

    scores_A = [
        r["hybrid_score"]
        for r in hybrid_results_A_05[qid]
    ]

    scores_B = [
        r["hybrid_score"]
        for r in hybrid_results_B_05[qid]
    ]

    assert scores_A == sorted(scores_A, reverse=True)
    assert scores_B == sorted(scores_B, reverse=True)

print("Top-10 results available for all questions ✓")
print("Hybrid scores sorted correctly ✓")
print("STEP 7.4.2 PASSED — HYBRID α=0.5 READY ✓")

HYBRID RETRIEVAL — α = 0.5
Experiment A: 12 questions ✓
Experiment B: 12 questions ✓
Top-10 results available for all questions ✓
Hybrid scores sorted correctly ✓
STEP 7.4.2 PASSED — HYBRID α=0.5 READY ✓


# STEP 7.5 — Hybrid Evaluation (α = 0.5)

The baseline hybrid configuration (α = 0.5) is evaluated using the same
fixed benchmark and ground-truth matching procedure used for BM25.

The evaluation metrics are:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- Mean Reciprocal Rank (MRR)

Results are calculated independently for Experiment A and Experiment B.

This allows a direct comparison between the hybrid approach and the
previous BM25 baseline without changing the evaluation methodology.

## STEP 7.5.1 — Calculate Hybrid Metrics

Hybrid retrieval is evaluated question-by-question using the same
Recall@K and MRR definitions used for the BM25 evaluation.

The existing evaluation functions operate on the results of a single
question. Therefore, the benchmark-level calculation applies these
functions independently to each question and then averages the results.

This preserves the exact evaluation methodology used for the BM25
baseline.

In [53]:
# STEP 7.5.1 — Hybrid Metrics Calculation

def evaluate_retrieval_results(
    retrieval_results,
    benchmark
):
    """
    Calculate Recall@K and MRR across the full benchmark.

    Uses the existing question-level evaluation functions.
    """

    metrics = {}

    for k in K_VALUES:

        recall_values = []

        for question in benchmark:

            qid = question["id"]
            expected_pages = question["expected_pages"]

            results = retrieval_results[qid]

            recall = calculate_recall_at_k(
                results,
                expected_pages,
                k
            )

            recall_values.append(recall)

        metrics[f"Recall@{k}"] = (
            sum(recall_values) / len(recall_values)
        )

    # -----------------------------
    # MRR
    # -----------------------------

    reciprocal_ranks = []

    for question in benchmark:

        qid = question["id"]
        expected_pages = question["expected_pages"]

        results = retrieval_results[qid]

        rr = calculate_reciprocal_rank(
            results,
            expected_pages
        )

        reciprocal_ranks.append(rr)

    metrics["MRR"] = (
        sum(reciprocal_ranks) / len(reciprocal_ranks)
    )

    return metrics


# ==================================================
# Evaluate Experiment A
# ==================================================

hybrid_metrics_A_05 = evaluate_retrieval_results(
    hybrid_results_A_05,
    evaluation_benchmark
)


# ==================================================
# Evaluate Experiment B
# ==================================================

hybrid_metrics_B_05 = evaluate_retrieval_results(
    hybrid_results_B_05,
    evaluation_benchmark
)


# ==================================================
# Display
# ==================================================

print("=" * 70)
print("HYBRID METRICS — α = 0.5")
print("=" * 70)

print("\nExperiment A")

for metric, value in hybrid_metrics_A_05.items():
    print(f"{metric:<12}: {value:.4f}")


print("\nExperiment B")

for metric, value in hybrid_metrics_B_05.items():
    print(f"{metric:<12}: {value:.4f}")


print("\nSTEP 7.5.1 PASSED — HYBRID METRICS CALCULATED ✓")

HYBRID METRICS — α = 0.5

Experiment A
Recall@1    : 0.0833
Recall@3    : 0.0833
Recall@5    : 0.2500
Recall@10   : 0.4167
MRR         : 0.1396

Experiment B
Recall@1    : 0.0833
Recall@3    : 0.2500
Recall@5    : 0.2500
Recall@10   : 0.4167
MRR         : 0.1750

STEP 7.5.1 PASSED — HYBRID METRICS CALCULATED ✓


## STEP 7.5.2 — Hybrid vs BM25 Baseline

The baseline hybrid configuration (α = 0.5) is compared directly with
the BM25 baseline using the same evaluation benchmark and metrics.

The comparison is performed independently for Experiments A and B.

The purpose is to determine whether combining semantic and lexical
retrieval improves ranking quality over BM25 alone.

The comparison uses:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR

No changes are made to the benchmark or ground-truth definitions.

In [54]:
# STEP 7.5.2 — Compare Hybrid α=0.5 vs BM25

print("=" * 90)
print("HYBRID α=0.5 vs BM25")
print("=" * 90)

print(
    f"{'Metric':<12}"
    f"{'BM25 A':<12}"
    f"{'Hybrid A':<12}"
    f"{'Δ A':<12}"
    f"{'BM25 B':<12}"
    f"{'Hybrid B':<12}"
    f"{'Δ B':<12}"
)

print("-" * 90)

for metric in ["Recall@1", "Recall@3", "Recall@5", "Recall@10", "MRR"]:

    bm25_A = bm25_metrics_A[metric]
    hybrid_A = hybrid_metrics_A_05[metric]

    bm25_B = bm25_metrics_B[metric]
    hybrid_B = hybrid_metrics_B_05[metric]

    delta_A = hybrid_A - bm25_A
    delta_B = hybrid_B - bm25_B

    print(
        f"{metric:<12}"
        f"{bm25_A:<12.4f}"
        f"{hybrid_A:<12.4f}"
        f"{delta_A:+.4f}     "
        f"{bm25_B:<12.4f}"
        f"{hybrid_B:<12.4f}"
        f"{delta_B:+.4f}"
    )


print("\n" + "=" * 90)
print("COMPARISON COMPLETED")
print("=" * 90)

HYBRID α=0.5 vs BM25
Metric      BM25 A      Hybrid A    Δ A         BM25 B      Hybrid B    Δ B         
------------------------------------------------------------------------------------------
Recall@1    0.0000      0.0833      +0.0833     0.0000      0.0833      +0.0833
Recall@3    0.0000      0.0833      +0.0833     0.0833      0.2500      +0.1667
Recall@5    0.1667      0.2500      +0.0833     0.1667      0.2500      +0.0833
Recall@10   0.2500      0.4167      +0.1667     0.3333      0.4167      +0.0833
MRR         0.0452      0.1396      +0.0943     0.0671      0.1750      +0.1079

COMPARISON COMPLETED


# STEP 7.6 — Hybrid Weight Tuning

The hybrid retrieval system is evaluated using multiple values of α
to determine the optimal balance between semantic and lexical retrieval.

The following configurations are tested:

- α = 0.25 → BM25-dominant
- α = 0.50 → Balanced
- α = 0.75 → Semantic-dominant

For each α, the system is evaluated on both chunking experiments:

- Experiment A: 400–600 tokens with overlap.
- Experiment B: 700–900 tokens.

The same fixed benchmark, ground-truth candidates, Recall@K metrics,
and MRR are used for every configuration.

The best configuration will be selected based on retrieval quality,
with particular attention to MRR and Recall@10.

No benchmark questions, ground-truth definitions, or evaluation
procedures are changed during weight tuning.

In [55]:
# STEP 7.6.1 — Run Hybrid Weight Tuning

ALPHA_VALUES = [0.25, 0.50, 0.75]

hybrid_results_by_alpha_A = {}
hybrid_results_by_alpha_B = {}

for alpha in ALPHA_VALUES:

    print("=" * 70)
    print(f"BUILDING HYBRID RESULTS — α = {alpha}")
    print("=" * 70)

    results_A = {}
    results_B = {}

    for question in evaluation_benchmark:

        qid = question["id"]

        results_A[qid] = build_hybrid_results(
            bm25_results=normalized_bm25_A[qid],
            semantic_results=normalized_semantic_A[qid],
            alpha=alpha,
            top_k=10
        )

        results_B[qid] = build_hybrid_results(
            bm25_results=normalized_bm25_B[qid],
            semantic_results=normalized_semantic_B[qid],
            alpha=alpha,
            top_k=10
        )

    hybrid_results_by_alpha_A[alpha] = results_A
    hybrid_results_by_alpha_B[alpha] = results_B

    # Validation
    assert len(results_A) == len(evaluation_benchmark)
    assert len(results_B) == len(evaluation_benchmark)

    for qid in results_A:

        assert len(results_A[qid]) == 10
        assert len(results_B[qid]) == 10

        scores_A = [
            r["hybrid_score"]
            for r in results_A[qid]
        ]

        scores_B = [
            r["hybrid_score"]
            for r in results_B[qid]
        ]

        assert scores_A == sorted(scores_A, reverse=True)
        assert scores_B == sorted(scores_B, reverse=True)

    print(f"Experiment A: {len(results_A)} questions ✓")
    print(f"Experiment B: {len(results_B)} questions ✓")
    print("Top-10 ranking validated ✓")


print("\n" + "=" * 70)
print("STEP 7.6.1 PASSED — ALL HYBRID WEIGHTS READY ✓")
print("=" * 70)

BUILDING HYBRID RESULTS — α = 0.25
Experiment A: 12 questions ✓
Experiment B: 12 questions ✓
Top-10 ranking validated ✓
BUILDING HYBRID RESULTS — α = 0.5
Experiment A: 12 questions ✓
Experiment B: 12 questions ✓
Top-10 ranking validated ✓
BUILDING HYBRID RESULTS — α = 0.75
Experiment A: 12 questions ✓
Experiment B: 12 questions ✓
Top-10 ranking validated ✓

STEP 7.6.1 PASSED — ALL HYBRID WEIGHTS READY ✓


## STEP 7.6.2 — Evaluate Hybrid Weight Configurations

Each hybrid weight configuration is evaluated using the fixed benchmark
and the same retrieval metrics used in previous experiments.

Tested configurations:

- α = 0.25: BM25-dominant
- α = 0.50: Balanced
- α = 0.75: Semantic-dominant

For every configuration and chunking experiment, we calculate:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR

The evaluation procedure and ground-truth matching remain unchanged
to ensure a fair comparison between all configurations.

In [56]:
# STEP 7.6.2 — Evaluate All Hybrid Weights

hybrid_metrics_by_alpha_A = {}
hybrid_metrics_by_alpha_B = {}

for alpha in ALPHA_VALUES:

    hybrid_metrics_by_alpha_A[alpha] = evaluate_retrieval_results(
        hybrid_results_by_alpha_A[alpha],
        evaluation_benchmark
    )

    hybrid_metrics_by_alpha_B[alpha] = evaluate_retrieval_results(
        hybrid_results_by_alpha_B[alpha],
        evaluation_benchmark
    )


# ==================================================
# Display Results
# ==================================================

print("=" * 95)
print("HYBRID WEIGHT TUNING RESULTS")
print("=" * 95)

print(
    f"{'α':<8}"
    f"{'R@1 A':<10}"
    f"{'R@3 A':<10}"
    f"{'R@5 A':<10}"
    f"{'R@10 A':<11}"
    f"{'MRR A':<10}"
    f"{'R@1 B':<10}"
    f"{'R@3 B':<10}"
    f"{'R@5 B':<10}"
    f"{'R@10 B':<11}"
    f"{'MRR B':<10}"
)

print("-" * 95)

for alpha in ALPHA_VALUES:

    A = hybrid_metrics_by_alpha_A[alpha]
    B = hybrid_metrics_by_alpha_B[alpha]

    print(
        f"{alpha:<8.2f}"
        f"{A['Recall@1']:<10.4f}"
        f"{A['Recall@3']:<10.4f}"
        f"{A['Recall@5']:<10.4f}"
        f"{A['Recall@10']:<11.4f}"
        f"{A['MRR']:<10.4f}"
        f"{B['Recall@1']:<10.4f}"
        f"{B['Recall@3']:<10.4f}"
        f"{B['Recall@5']:<10.4f}"
        f"{B['Recall@10']:<11.4f}"
        f"{B['MRR']:<10.4f}"
    )


print("\nSTEP 7.6.2 PASSED — ALL HYBRID WEIGHTS EVALUATED ✓")

HYBRID WEIGHT TUNING RESULTS
α       R@1 A     R@3 A     R@5 A     R@10 A     MRR A     R@1 B     R@3 B     R@5 B     R@10 B     MRR B     
-----------------------------------------------------------------------------------------------
0.25    0.0000    0.0833    0.1667    0.3333     0.0688    0.0833    0.1667    0.1667    0.3333     0.1369    
0.50    0.0833    0.0833    0.2500    0.4167     0.1396    0.0833    0.2500    0.2500    0.4167     0.1750    
0.75    0.0833    0.1667    0.3333    0.4167     0.1667    0.1667    0.2500    0.2500    0.5000     0.2239    

STEP 7.6.2 PASSED — ALL HYBRID WEIGHTS EVALUATED ✓


## STEP 7.6.3 — Select Best Hybrid Weight

The three hybrid configurations are compared using the benchmark
retrieval metrics.

The primary selection criterion is MRR because it evaluates how early
the first relevant evidence is retrieved.

Recall@10 is used as a secondary criterion because the RAG pipeline
must retrieve relevant evidence within the final candidate set.

Based on the current results:

- α = 0.25 is the weakest configuration.
- α = 0.50 improves retrieval quality.
- α = 0.75 achieves the highest MRR for both experiments.
- α = 0.75 also achieves the highest Recall@10 for Experiment B.

Therefore, α = 0.75 is selected as the current best hybrid weight.

This configuration will be used as the baseline for the next retrieval
stage: Cross-Encoder Reranking.

The selected configuration is not considered final until it is compared
with the reranked hybrid pipeline.

In [57]:
# STEP 7.6.3 — Select Best Hybrid Weight

# Primary criterion: MRR
best_alpha_A = max(
    ALPHA_VALUES,
    key=lambda alpha: hybrid_metrics_by_alpha_A[alpha]["MRR"]
)

best_alpha_B = max(
    ALPHA_VALUES,
    key=lambda alpha: hybrid_metrics_by_alpha_B[alpha]["MRR"]
)


print("=" * 80)
print("BEST HYBRID WEIGHT SELECTION")
print("=" * 80)

print(
    f"Experiment A best α: {best_alpha_A:.2f} "
    f"(MRR = {hybrid_metrics_by_alpha_A[best_alpha_A]['MRR']:.4f})"
)

print(
    f"Experiment B best α: {best_alpha_B:.2f} "
    f"(MRR = {hybrid_metrics_by_alpha_B[best_alpha_B]['MRR']:.4f})"
)


# --------------------------------------------------
# Check whether both experiments agree
# --------------------------------------------------

if best_alpha_A == best_alpha_B:

    BEST_ALPHA = best_alpha_A

    print(
        f"\nSelected global best α: {BEST_ALPHA:.2f} ✓"
    )

else:

    # If experiments disagree, use the average MRR
    BEST_ALPHA = max(
        ALPHA_VALUES,
        key=lambda alpha: (
            hybrid_metrics_by_alpha_A[alpha]["MRR"]
            + hybrid_metrics_by_alpha_B[alpha]["MRR"]
        ) / 2
    )

    print(
        f"\nExperiments disagree."
        f"\nSelected α based on average MRR: {BEST_ALPHA:.2f} ✓"
    )


print("\n" + "=" * 80)
print("SELECTED HYBRID CONFIGURATION")
print("=" * 80)

print(f"α = {BEST_ALPHA:.2f}")
print(f"Semantic contribution = {BEST_ALPHA:.2f}")
print(f"BM25 contribution = {1 - BEST_ALPHA:.2f}")

print("\nSTEP 7.6.3 PASSED — BEST HYBRID WEIGHT SELECTED ✓")

BEST HYBRID WEIGHT SELECTION
Experiment A best α: 0.75 (MRR = 0.1667)
Experiment B best α: 0.75 (MRR = 0.2239)

Selected global best α: 0.75 ✓

SELECTED HYBRID CONFIGURATION
α = 0.75
Semantic contribution = 0.75
BM25 contribution = 0.25

STEP 7.6.3 PASSED — BEST HYBRID WEIGHT SELECTED ✓


# STEP 7.7 — Save Best Hybrid Configuration

The best hybrid configuration identified during weight tuning is saved
as a reusable retrieval artifact.

Selected configuration:

- α = 0.75
- Semantic contribution = 75%
- BM25 contribution = 25%

The selected configuration is consistent across both chunking
experiments.

The best hybrid retrieval results and evaluation metrics are saved for
later comparison with the Cross-Encoder reranking stage.

This ensures reproducibility and prevents unnecessary recomputation.

In [58]:
# STEP 7.7 — Save Best Hybrid Configuration

import json
import os

processed_dir = r"C:\Users\Rahma mohamed\OneDrive\Desktop\AI-Portfolio\7__CardioPress AI\Data\processed"

os.makedirs(processed_dir, exist_ok=True)


# ==================================================
# Save best configuration
# ==================================================

best_hybrid_config = {
    "best_alpha": BEST_ALPHA,
    "semantic_weight": BEST_ALPHA,
    "bm25_weight": 1.0 - BEST_ALPHA,
    "experiment_A": {
        "best_alpha": best_alpha_A,
        "metrics": hybrid_metrics_by_alpha_A[best_alpha_A]
    },
    "experiment_B": {
        "best_alpha": best_alpha_B,
        "metrics": hybrid_metrics_by_alpha_B[best_alpha_B]
    }
}


config_path = os.path.join(
    processed_dir,
    "best_hybrid_config.json"
)


with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_hybrid_config,
        f,
        indent=4
    )


# ==================================================
# Prepare best hybrid results
# ==================================================

best_hybrid_results_A = hybrid_results_by_alpha_A[BEST_ALPHA]
best_hybrid_results_B = hybrid_results_by_alpha_B[BEST_ALPHA]


print("=" * 80)
print("BEST HYBRID CONFIGURATION SAVED")
print("=" * 80)

print(f"Best α: {BEST_ALPHA}")
print(f"Semantic weight: {BEST_ALPHA}")
print(f"BM25 weight: {1 - BEST_ALPHA}")

print(f"\nConfiguration file:")
print(config_path)

print("\nBest Hybrid Results:")
print(f"Experiment A: {len(best_hybrid_results_A)} questions ✓")
print(f"Experiment B: {len(best_hybrid_results_B)} questions ✓")

print("\nSTEP 7.7 PASSED — BEST HYBRID CONFIGURATION SAVED ✓")

BEST HYBRID CONFIGURATION SAVED
Best α: 0.75
Semantic weight: 0.75
BM25 weight: 0.25

Configuration file:
C:\Users\Rahma mohamed\OneDrive\Desktop\AI-Portfolio\7__CardioPress AI\Data\processed\best_hybrid_config.json

Best Hybrid Results:
Experiment A: 12 questions ✓
Experiment B: 12 questions ✓

STEP 7.7 PASSED — BEST HYBRID CONFIGURATION SAVED ✓


_________________________________________

____________________________________

# STEP 7.8 — Cross-Encoder Reranker Setup


A Cross-Encoder reranker will be added after Hybrid retrieval to improve
the ordering of the retrieved candidate chunks.

Unlike bi-encoder semantic retrieval, which independently encodes the
query and document, a Cross-Encoder receives the query and candidate
chunk together and directly estimates their relevance.

The retrieval pipeline will therefore be:

Query
→ Semantic Retrieval
→ BM25 Retrieval
→ Hybrid Fusion (α = 0.75)
→ Candidate Selection
→ Cross-Encoder Reranking
→ Final Top-K Results

The Cross-Encoder will rerank only the candidate set produced by the
Hybrid retriever rather than the entire corpus.

The same fixed evaluation benchmark and ground-truth definitions will
be used.

The final evaluation will compare:

1. Semantic Retrieval
2. BM25 Retrieval
3. Hybrid Retrieval
4. Hybrid + Cross-Encoder Reranker

Evaluation metrics:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR

The goal is to determine whether Cross-Encoder reranking provides a
measurable improvement over the best Hybrid configuration.

________________________

## STEP 7.8.1 — Load & Validate Cross-Encoder Model

A Cross-Encoder model is loaded for query–chunk relevance scoring.

The model receives the query and candidate chunk together and produces
a relevance score for each pair.

The model is validated before integration into the retrieval pipeline.

Validation checks include:

- Model loads successfully.
- Tokenizer loads successfully.
- Query–chunk pairs can be encoded.
- The model produces a finite relevance score.
- The model can process a sample query and chunk without errors.

No benchmark evaluation is performed at this stage.

In [59]:
# STEP 7.8.1 — Load & Validate Cross-Encoder Model

from sentence_transformers import CrossEncoder
import numpy as np

CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print("=" * 80)
print("CROSS-ENCODER MODEL LOADING")
print("=" * 80)

print(f"Model: {CROSS_ENCODER_NAME}")

cross_encoder = CrossEncoder(
    CROSS_ENCODER_NAME
)

print("Cross-Encoder loaded successfully ✓")


# ==================================================
# Sample validation
# ==================================================

test_query = "How does reducing salt intake help prevent hypertension?"

test_chunk = chunks_A[0]["text"]

test_pair = [
    [test_query, test_chunk]
]


test_score = cross_encoder.predict(
    test_pair
)


test_score = float(
    np.asarray(test_score).reshape(-1)[0]
)


# ==================================================
# Validation
# ==================================================

assert np.isfinite(test_score), \
    "Cross-Encoder produced a non-finite score."

print("\n" + "=" * 80)
print("CROSS-ENCODER VALIDATION")
print("=" * 80)

print(f"Query–chunk pair encoded ✓")
print(f"Relevance score: {test_score:.6f}")
print("Score is finite ✓")

print("\nSTEP 7.8.1 PASSED — CROSS-ENCODER VALIDATED ✓")

CROSS-ENCODER MODEL LOADING
Model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4999.06it/s]


Cross-Encoder loaded successfully ✓

CROSS-ENCODER VALIDATION
Query–chunk pair encoded ✓
Relevance score: -6.521044
Score is finite ✓

STEP 7.8.1 PASSED — CROSS-ENCODER VALIDATED ✓


## STEP 7.8.2 — Hybrid Candidate Pool Construction

The Cross-Encoder will operate as a reranker rather than a first-stage
retriever.

For each benchmark question, the best Hybrid retrieval configuration
(α = 0.75) is used to generate the initial candidate pool.

The candidate pool combines the retrieved evidence from the Hybrid
stage and provides a limited set of candidates for Cross-Encoder
reranking.

The candidate pool size is set to Top-10 to maintain a practical
two-stage retrieval architecture:

    Query
      ↓
    Hybrid Retrieval
      ↓
    Top-10 Candidates
      ↓
    Cross-Encoder
      ↓
    Reranked Top-10

The same candidate pool is used for the reranking evaluation so that
any performance improvement can be attributed to the Cross-Encoder's
ability to reorder candidates rather than retrieve new documents.

The candidate pool is validated for:

- All 12 benchmark questions.
- Exactly 10 candidates per question.
- Unique chunk IDs.
- Valid hybrid scores.

In [60]:
# STEP 7.8.2 — Hybrid Candidate Pool Validation

CANDIDATE_POOL_SIZE = 10

candidate_pool_A = {}
candidate_pool_B = {}

for question in evaluation_benchmark:

    qid = question["id"]

    candidate_pool_A[qid] = best_hybrid_results_A[qid][
        :CANDIDATE_POOL_SIZE
    ]

    candidate_pool_B[qid] = best_hybrid_results_B[qid][
        :CANDIDATE_POOL_SIZE
    ]


# ==================================================
# Validation
# ==================================================

assert len(candidate_pool_A) == 12
assert len(candidate_pool_B) == 12

for qid in candidate_pool_A:

    candidates_A = candidate_pool_A[qid]
    candidates_B = candidate_pool_B[qid]

    # Exactly 10 candidates
    assert len(candidates_A) == CANDIDATE_POOL_SIZE
    assert len(candidates_B) == CANDIDATE_POOL_SIZE

    # Unique chunk IDs
    ids_A = [x["chunk_id"] for x in candidates_A]
    ids_B = [x["chunk_id"] for x in candidates_B]

    assert len(ids_A) == len(set(ids_A))
    assert len(ids_B) == len(set(ids_B))

    # Valid hybrid scores
    assert all(
        np.isfinite(x["hybrid_score"])
        for x in candidates_A
    )

    assert all(
        np.isfinite(x["hybrid_score"])
        for x in candidates_B
    )


print("=" * 80)
print("HYBRID CANDIDATE POOL VALIDATION")
print("=" * 80)

print(f"Candidate pool size: {CANDIDATE_POOL_SIZE}")
print(f"Experiment A: {len(candidate_pool_A)} questions ✓")
print(f"Experiment B: {len(candidate_pool_B)} questions ✓")
print("Exactly 10 candidates per question ✓")
print("Chunk IDs unique ✓")
print("Hybrid scores valid ✓")

print("\nSTEP 7.8.2 PASSED — CANDIDATE POOLS READY ✓")

HYBRID CANDIDATE POOL VALIDATION
Candidate pool size: 10
Experiment A: 12 questions ✓
Experiment B: 12 questions ✓
Exactly 10 candidates per question ✓
Chunk IDs unique ✓
Hybrid scores valid ✓

STEP 7.8.2 PASSED — CANDIDATE POOLS READY ✓


## STEP 7.8.3 — Cross-Encoder Reranking

The validated Hybrid Top-10 candidate pool is reranked using the
Cross-Encoder model.

For each benchmark question, the Cross-Encoder receives pairs of:

    [Query, Candidate Chunk]

The model produces a relevance score for every query–chunk pair.

Candidates are then sorted in descending order of the Cross-Encoder
score.

The Cross-Encoder does not introduce new chunks. It only reorders the
existing Hybrid candidate pool.

Reranking is performed independently for:

- Experiment A
- Experiment B

The following information is preserved for every candidate:

- Chunk ID
- Section title
- Page numbers
- Token count
- Original Hybrid score
- Cross-Encoder score
- Original Hybrid rank
- New Cross-Encoder rank

This allows the final system to be compared fairly with the original
Hybrid retrieval stage.

In [61]:
# STEP 7.8.3 — Cross-Encoder Reranking

def rerank_with_cross_encoder(
    query,
    candidates,
    cross_encoder
):
    """
    Rerank Hybrid candidates using a Cross-Encoder.

    The Cross-Encoder only reorders the existing candidates.
    """

    # --------------------------------------------------
    # Build query-chunk pairs
    # --------------------------------------------------

    pairs = [
        [query, candidate["text"]]
        for candidate in candidates
    ]

    # --------------------------------------------------
    # Predict relevance scores
    # --------------------------------------------------

    scores = cross_encoder.predict(
        pairs,
        show_progress_bar=False
    )

    scores = np.asarray(scores).reshape(-1)

    # --------------------------------------------------
    # Validate scores
    # --------------------------------------------------

    assert len(scores) == len(candidates)

    assert np.all(
        np.isfinite(scores)
    )

    # --------------------------------------------------
    # Build reranked results
    # --------------------------------------------------

    reranked = []

    for original_rank, (candidate, score) in enumerate(
        zip(candidates, scores),
        start=1
    ):

        item = candidate.copy()

        item["hybrid_rank"] = original_rank
        item["cross_encoder_score"] = float(score)

        reranked.append(item)

    # --------------------------------------------------
    # Sort by Cross-Encoder score
    # --------------------------------------------------

    reranked.sort(
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )

    # --------------------------------------------------
    # Assign new rank
    # --------------------------------------------------

    for rank, result in enumerate(
        reranked,
        start=1
    ):
        result["reranker_rank"] = rank

    return reranked


# ==================================================
# Run reranking
# ==================================================

cross_encoder_results_A = {}
cross_encoder_results_B = {}

for question in evaluation_benchmark:

    qid = question["id"]
    query = question["query"]

    cross_encoder_results_A[qid] = rerank_with_cross_encoder(
        query=query,
        candidates=candidate_pool_A[qid],
        cross_encoder=cross_encoder
    )

    cross_encoder_results_B[qid] = rerank_with_cross_encoder(
        query=query,
        candidates=candidate_pool_B[qid],
        cross_encoder=cross_encoder
    )


# ==================================================
# Validation
# ==================================================

for qid in cross_encoder_results_A:

    results_A = cross_encoder_results_A[qid]
    results_B = cross_encoder_results_B[qid]

    assert len(results_A) == 10
    assert len(results_B) == 10

    scores_A = [
        r["cross_encoder_score"]
        for r in results_A
    ]

    scores_B = [
        r["cross_encoder_score"]
        for r in results_B
    ]

    # Scores must be finite
    assert all(
        np.isfinite(score)
        for score in scores_A
    )

    assert all(
        np.isfinite(score)
        for score in scores_B
    )

    # Scores must be descending
    assert scores_A == sorted(
        scores_A,
        reverse=True
    )

    assert scores_B == sorted(
        scores_B,
        reverse=True
    )

    # No chunks added or removed
    hybrid_ids_A = {
        x["chunk_id"]
        for x in candidate_pool_A[qid]
    }

    reranked_ids_A = {
        x["chunk_id"]
        for x in results_A
    }

    hybrid_ids_B = {
        x["chunk_id"]
        for x in candidate_pool_B[qid]
    }

    reranked_ids_B = {
        x["chunk_id"]
        for x in results_B
    }

    assert hybrid_ids_A == reranked_ids_A
    assert hybrid_ids_B == reranked_ids_B


print("=" * 80)
print("CROSS-ENCODER RERANKING")
print("=" * 80)

print(f"Experiment A: {len(cross_encoder_results_A)} questions ✓")
print(f"Experiment B: {len(cross_encoder_results_B)} questions ✓")
print("10 candidates reranked per question ✓")
print("Scores finite ✓")
print("Scores sorted descending ✓")
print("No candidates added or removed ✓")

print("\nSTEP 7.8.3 PASSED — CROSS-ENCODER RERANKING READY ✓")

CROSS-ENCODER RERANKING
Experiment A: 12 questions ✓
Experiment B: 12 questions ✓
10 candidates reranked per question ✓
Scores finite ✓
Scores sorted descending ✓
No candidates added or removed ✓

STEP 7.8.3 PASSED — CROSS-ENCODER RERANKING READY ✓


## STEP 7.8.4 — Cross-Encoder Ranking Inspection

Before calculating the final evaluation metrics, selected benchmark
questions are inspected to verify the behavior of the Cross-Encoder.

The inspection compares:

- Original Hybrid rank
- Cross-Encoder rank
- Hybrid score
- Cross-Encoder score
- Chunk ID
- Section
- Pages

Q01 and Q02 are inspected because they represent direct clinical
questions with expected evidence in the dataset.

This is a qualitative sanity check only.

No metrics or model-selection decisions are made at this stage.

In [62]:
# STEP 7.8.4 — Cross-Encoder Ranking Inspection

INSPECTION_QUESTIONS = ["Q01", "Q02"]

for qid in INSPECTION_QUESTIONS:

    question = next(
        q for q in evaluation_benchmark
        if q["id"] == qid
    )

    print("\n" + "=" * 90)
    print(f"CROSS-ENCODER INSPECTION — {qid}")
    print("=" * 90)

    print(f"Query: {question['query']}")
    print(f"Expected section: {question['expected_section']}")
    print(f"Expected pages: {question['expected_pages']}")

    print("\n" + "-" * 90)
    print("RERANKED RESULTS")
    print("-" * 90)

    results = cross_encoder_results_A[qid]

    for result in results:

        print(
            f"CE Rank: {result['reranker_rank']:>2} | "
            f"Hybrid Rank: {result['hybrid_rank']:>2} | "
            f"Chunk: {result['chunk_id']} | "
            f"Pages: {result['pages']} | "
            f"Hybrid: {result['hybrid_score']:.4f} | "
            f"CE: {result['cross_encoder_score']:.4f}"
        )

print("\n" + "=" * 90)
print("STEP 7.8.4 COMPLETED — RANKING INSPECTION READY ✓")
print("=" * 90)


CROSS-ENCODER INSPECTION — Q01
Query: How does reducing salt intake help prevent hypertension?
Expected section: Module 2.4 — Healthy Diet
Expected pages: [149, 150, 151, 152, 153]

------------------------------------------------------------------------------------------
RERANKED RESULTS
------------------------------------------------------------------------------------------
CE Rank:  1 | Hybrid Rank:  1 | Chunk: A_0007 | Pages: [161, 162, 163] | Hybrid: 0.9404 | CE: -1.1004
CE Rank:  2 | Hybrid Rank:  4 | Chunk: A_0071 | Pages: [320, 321] | Hybrid: 0.4244 | CE: -5.2775
CE Rank:  3 | Hybrid Rank:  6 | Chunk: A_0050 | Pages: [258, 259] | Hybrid: 0.3078 | CE: -6.1755
CE Rank:  4 | Hybrid Rank:  7 | Chunk: A_0059 | Pages: [287, 288, 289] | Hybrid: 0.2919 | CE: -6.2399
CE Rank:  5 | Hybrid Rank:  9 | Chunk: A_0005 | Pages: [158, 159, 160] | Hybrid: 0.2504 | CE: -6.3400
CE Rank:  6 | Hybrid Rank:  2 | Chunk: A_0047 | Pages: [254, 255, 256] | Hybrid: 0.7500 | CE: -6.5506
CE Rank:  7 | Hy

## STEP 7.8.5 — Cross-Encoder Evaluation

The Cross-Encoder reranked results are now evaluated using the same
fixed benchmark and ground-truth definitions used for Semantic, BM25,
and Hybrid retrieval.

No changes are made to:

- Benchmark questions
- Expected evidence
- Ground-truth matching
- Evaluation metrics

The evaluated metrics are:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- Mean Reciprocal Rank (MRR)

The evaluation is performed independently for:

- Experiment A
- Experiment B

The purpose of this step is to measure whether Cross-Encoder reranking
improves the retrieval quality of the best Hybrid configuration
(α = 0.75).

The Cross-Encoder can only reorder the Hybrid candidate pool. Therefore,
it cannot introduce evidence that was not already retrieved by Hybrid
Top-10.

In [63]:
# STEP 7.8.5 — Cross-Encoder Evaluation

K_VALUES = [1, 3, 5, 10]


# ============================================================
# Helper: Recall@K
# ============================================================

def calculate_ce_recall_at_k(
    reranked_results,
    benchmark,
    k
):
    """
    Calculate Recall@K for Cross-Encoder reranked results.
    """

    recalls = []

    for question in benchmark:

        qid = question["id"]

        expected_pages = question["expected_pages"]

        results = reranked_results[qid][:k]

        # Out-of-scope questions
        if question["expected_evidence"] is None:
            recalls.append(1 if len(results) > 0 else 0)
            continue

        found = False

        for result in results:

            if is_relevant(
                result,
                expected_pages
            ):
                found = True
                break

        recalls.append(
            1 if found else 0
        )

    return float(
        np.mean(recalls)
    )


# ============================================================
# Helper: MRR
# ============================================================

def calculate_ce_mrr(
    reranked_results,
    benchmark
):
    """
    Calculate Mean Reciprocal Rank for Cross-Encoder results.
    """

    reciprocal_ranks = []

    for question in benchmark:

        qid = question["id"]

        expected_pages = question["expected_pages"]

        results = reranked_results[qid]

        # Out-of-scope
        if question["expected_evidence"] is None:
            reciprocal_ranks.append(0.0)
            continue

        first_relevant_rank = None

        for rank, result in enumerate(
            results,
            start=1
        ):

            if is_relevant(
                result,
                expected_pages
            ):
                first_relevant_rank = rank
                break

        if first_relevant_rank is None:
            reciprocal_ranks.append(0.0)

        else:
            reciprocal_ranks.append(
                1.0 / first_relevant_rank
            )

    return float(
        np.mean(reciprocal_ranks)
    )


# ============================================================
# Calculate Experiment A
# ============================================================

cross_encoder_metrics_A = {}

for k in K_VALUES:

    cross_encoder_metrics_A[
        f"Recall@{k}"
    ] = calculate_ce_recall_at_k(
        cross_encoder_results_A,
        evaluation_benchmark,
        k
    )

cross_encoder_metrics_A["MRR"] = calculate_ce_mrr(
    cross_encoder_results_A,
    evaluation_benchmark
)


# ============================================================
# Calculate Experiment B
# ============================================================

cross_encoder_metrics_B = {}

for k in K_VALUES:

    cross_encoder_metrics_B[
        f"Recall@{k}"
    ] = calculate_ce_recall_at_k(
        cross_encoder_results_B,
        evaluation_benchmark,
        k
    )

cross_encoder_metrics_B["MRR"] = calculate_ce_mrr(
    cross_encoder_results_B,
    evaluation_benchmark
)


# ============================================================
# Display
# ============================================================

print("=" * 80)
print("CROSS-ENCODER METRICS")
print("=" * 80)

print("\nExperiment A")

for metric, value in cross_encoder_metrics_A.items():
    print(
        f"{metric:<12}: {value:.4f}"
    )


print("\nExperiment B")

for metric, value in cross_encoder_metrics_B.items():
    print(
        f"{metric:<12}: {value:.4f}"
    )


print(
    "\nSTEP 7.8.5 PASSED — "
    "CROSS-ENCODER METRICS CALCULATED ✓"
)

CROSS-ENCODER METRICS

Experiment A
Recall@1    : 0.2500
Recall@3    : 0.2500
Recall@5    : 0.2500
Recall@10   : 0.5833
MRR         : 0.1299

Experiment B
Recall@1    : 0.2500
Recall@3    : 0.2500
Recall@5    : 0.5833
Recall@10   : 0.6667
MRR         : 0.1708

STEP 7.8.5 PASSED — CROSS-ENCODER METRICS CALCULATED ✓


## STEP 7.8.6 — Hybrid vs Cross-Encoder Comparison

The Cross-Encoder reranking results are compared directly with the
selected Hybrid configuration (α = 0.75).

The comparison uses the same evaluation benchmark and the same
ground-truth matching procedure.

Metrics compared:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR

For each metric, the absolute change introduced by Cross-Encoder
reranking is calculated.

Positive change indicates an improvement, while negative change
indicates a degradation.

The comparison is performed separately for Experiment A and
Experiment B.

In [64]:
# STEP 7.8.6 — Hybrid vs Cross-Encoder Comparison

print("=" * 100)
print("HYBRID α=0.75 vs HYBRID + CROSS-ENCODER")
print("=" * 100)

print(
    f"{'Metric':<12}"
    f"{'Hybrid A':>12}"
    f"{'CE A':>12}"
    f"{'Δ A':>12}"
    f"{'Hybrid B':>12}"
    f"{'CE B':>12}"
    f"{'Δ B':>12}"
)

print("-" * 100)

for metric in [
    "Recall@1",
    "Recall@3",
    "Recall@5",
    "Recall@10",
    "MRR"
]:

    hybrid_A = hybrid_metrics_by_alpha_A[
        BEST_ALPHA
    ][metric]

    hybrid_B = hybrid_metrics_by_alpha_B[
        BEST_ALPHA
    ][metric]

    ce_A = cross_encoder_metrics_A[metric]
    ce_B = cross_encoder_metrics_B[metric]

    delta_A = ce_A - hybrid_A
    delta_B = ce_B - hybrid_B

    print(
        f"{metric:<12}"
        f"{hybrid_A:>12.4f}"
        f"{ce_A:>12.4f}"
        f"{delta_A:>+12.4f}"
        f"{hybrid_B:>12.4f}"
        f"{ce_B:>12.4f}"
        f"{delta_B:>+12.4f}"
    )

print("\n" + "=" * 100)
print("COMPARISON COMPLETED ✓")
print("=" * 100)

HYBRID α=0.75 vs HYBRID + CROSS-ENCODER
Metric          Hybrid A        CE A         Δ A    Hybrid B        CE B         Δ B
----------------------------------------------------------------------------------------------------
Recall@1          0.0833      0.2500     +0.1667      0.1667      0.2500     +0.0833
Recall@3          0.1667      0.2500     +0.0833      0.2500      0.2500     +0.0000
Recall@5          0.3333      0.2500     -0.0833      0.2500      0.5833     +0.3333
Recall@10         0.4167      0.5833     +0.1667      0.5000      0.6667     +0.1667
MRR               0.1667      0.1299     -0.0368      0.2239      0.1708     -0.0531

COMPARISON COMPLETED ✓


## STEP 7.8.7 — Cross-Encoder Performance by Question Type

Cross-Encoder performance is analyzed separately for each benchmark
question type.

The benchmark contains:

- Direct questions
- Paraphrased questions
- Abbreviation-based questions
- Threshold questions
- Out-of-scope questions

Recall@1, Recall@3, Recall@5, Recall@10, and MRR are calculated
per question type.

This analysis identifies which types of clinical questions benefit
from Cross-Encoder reranking and which types remain difficult.

The results will be used together with the overall metrics to interpret
the final retrieval performance.

In [65]:
# STEP 7.8.7 — Cross-Encoder Performance by Question Type

from collections import defaultdict

QUESTION_TYPES = [
    "direct",
    "paraphrased",
    "abbreviation",
    "threshold"
]


def calculate_type_metrics(
    reranked_results,
    benchmark,
    question_type
):

    questions = [
        q for q in benchmark
        if q["question_type"] == question_type
    ]

    if not questions:
        return {
            "Recall@1": 0.0,
            "Recall@3": 0.0,
            "Recall@5": 0.0,
            "Recall@10": 0.0,
            "MRR": 0.0
        }

    metrics = {}

    for k in [1, 3, 5, 10]:

        hits = []

        for question in questions:

            results = reranked_results[
                question["id"]
            ][:k]

            hit = any(
                is_relevant(
                    result,
                    question["expected_pages"]
                )
                for result in results
            )

            hits.append(
                1 if hit else 0
            )

        metrics[f"Recall@{k}"] = float(
            np.mean(hits)
        )

    # -----------------------------
    # MRR
    # -----------------------------

    reciprocal_ranks = []

    for question in questions:

        results = reranked_results[
            question["id"]
        ]

        rr = 0.0

        for rank, result in enumerate(
            results,
            start=1
        ):

            if is_relevant(
                result,
                question["expected_pages"]
            ):
                rr = 1.0 / rank
                break

        reciprocal_ranks.append(rr)

    metrics["MRR"] = float(
        np.mean(reciprocal_ranks)
    )

    return metrics


# ============================================================
# Experiment A
# ============================================================

ce_type_metrics_A = {}

for qtype in QUESTION_TYPES:

    ce_type_metrics_A[qtype] = calculate_type_metrics(
        cross_encoder_results_A,
        evaluation_benchmark,
        qtype
    )


# ============================================================
# Experiment B
# ============================================================

ce_type_metrics_B = {}

for qtype in QUESTION_TYPES:

    ce_type_metrics_B[qtype] = calculate_type_metrics(
        cross_encoder_results_B,
        evaluation_benchmark,
        qtype
    )


# ============================================================
# Display
# ============================================================

print("=" * 100)
print("CROSS-ENCODER PERFORMANCE BY QUESTION TYPE")
print("=" * 100)

print(
    f"{'Question Type':<16}"
    f"{'R@1 A':>10}"
    f"{'R@1 B':>10}"
    f"{'R@3 A':>10}"
    f"{'R@3 B':>10}"
    f"{'R@5 A':>10}"
    f"{'R@5 B':>10}"
    f"{'R@10 A':>11}"
    f"{'R@10 B':>11}"
)

print("-" * 100)

for qtype in QUESTION_TYPES:

    A = ce_type_metrics_A[qtype]
    B = ce_type_metrics_B[qtype]

    print(
        f"{qtype:<16}"
        f"{A['Recall@1']:>10.4f}"
        f"{B['Recall@1']:>10.4f}"
        f"{A['Recall@3']:>10.4f}"
        f"{B['Recall@3']:>10.4f}"
        f"{A['Recall@5']:>10.4f}"
        f"{B['Recall@5']:>10.4f}"
        f"{A['Recall@10']:>11.4f}"
        f"{B['Recall@10']:>11.4f}"
    )


print("\n" + "=" * 100)
print("MRR BY QUESTION TYPE")
print("=" * 100)

print(
    f"{'Question Type':<16}"
    f"{'MRR A':>12}"
    f"{'MRR B':>12}"
)

print("-" * 100)

for qtype in QUESTION_TYPES:

    print(
        f"{qtype:<16}"
        f"{ce_type_metrics_A[qtype]['MRR']:>12.4f}"
        f"{ce_type_metrics_B[qtype]['MRR']:>12.4f}"
    )


print(
    "\nSTEP 7.8.7 COMPLETED — "
    "QUESTION TYPE ANALYSIS ✓"
)

CROSS-ENCODER PERFORMANCE BY QUESTION TYPE
Question Type        R@1 A     R@1 B     R@3 A     R@3 B     R@5 A     R@5 B     R@10 A     R@10 B
----------------------------------------------------------------------------------------------------
direct              0.3333    0.3333    0.3333    0.3333    0.3333    0.3333     0.3333     0.3333
paraphrased         0.0000    0.0000    0.0000    0.0000    0.0000    0.5000     0.0000     0.5000
abbreviation        0.0000    0.0000    0.0000    0.0000    0.0000    0.5000     1.0000     1.0000
threshold           0.0000    0.0000    0.0000    0.0000    0.0000    0.6667     0.6667     0.6667

MRR BY QUESTION TYPE
Question Type          MRR A       MRR B
----------------------------------------------------------------------------------------------------
direct                0.3333      0.3333
paraphrased           0.0000      0.1250
abbreviation          0.1458      0.1750
threshold             0.0889      0.1500

STEP 7.8.7 COMPLETED — QUESTION 

## STEP 7.8.8 — Final Retrieval Comparison

All retrieval approaches are now compared using the same fixed
12-question evaluation benchmark and the same ground-truth matching
procedure.

The evaluated approaches are:

1. Semantic Retrieval
2. BM25 Retrieval
3. Hybrid Retrieval with α = 0.25
4. Hybrid Retrieval with α = 0.50
5. Hybrid Retrieval with α = 0.75
6. Hybrid Retrieval + Cross-Encoder Reranker

The comparison is performed separately for:

- Experiment A: 400–600 token chunks with 10% overlap
- Experiment B: 700–900 token chunks

Evaluation metrics:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR

The objective is to identify the strongest retrieval configuration
based on quantitative evidence rather than assuming that adding more
retrieval components always improves performance.

The selected configuration will be used as the retrieval pipeline for
the final RAG system.

In [66]:
# STEP 7.8.8.0 — Check Final Evaluation Variables

print("=" * 80)
print("FINAL EVALUATION VARIABLES")
print("=" * 80)

required_variables = [
    "semantic_metrics_A",
    "semantic_metrics_B",
    "bm25_metrics_A",
    "bm25_metrics_B",
    "hybrid_metrics_by_alpha_A",
    "hybrid_metrics_by_alpha_B",
    "cross_encoder_metrics_A",
    "cross_encoder_metrics_B",
    "BEST_ALPHA"
]

available = globals()

for name in required_variables:

    status = "✓" if name in available else "✗"

    print(
        f"{name:<30} {status}"
    )

print("=" * 80)

FINAL EVALUATION VARIABLES
semantic_metrics_A             ✗
semantic_metrics_B             ✗
bm25_metrics_A                 ✓
bm25_metrics_B                 ✓
hybrid_metrics_by_alpha_A      ✓
hybrid_metrics_by_alpha_B      ✓
cross_encoder_metrics_A        ✓
cross_encoder_metrics_B        ✓
BEST_ALPHA                     ✓


In [67]:
# STEP 7.8.8.1 — Locate Semantic Evaluation Variables

print("=" * 80)
print("SEMANTIC EVALUATION VARIABLES")
print("=" * 80)

semantic_variables = [
    name
    for name in list(globals().keys())
    if "semantic" in name.lower()
    and "metric" in name.lower()
]

if semantic_variables:
    for name in semantic_variables:
        value = globals()[name]

        print(
            f"{name:<40} "
            f"{type(value).__name__}"
        )

else:
    print("No semantic metric variables found.")

print("=" * 80)

SEMANTIC EVALUATION VARIABLES
No semantic metric variables found.


## STEP 7.8.8.2 — Rebuild Semantic Evaluation Metrics

The Semantic retrieval results were successfully generated for all
benchmark questions, but the final metric dictionaries were not
persisted under dedicated variables.

This step reconstructs the Semantic evaluation metrics directly from
the already-generated Semantic Top-10 results.

No retrieval is performed again.

The same fixed benchmark, ground-truth matching function, and evaluation
metrics used throughout the experiments are preserved.

Metrics:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR

The resulting metric dictionaries will be stored as:

- semantic_metrics_A
- semantic_metrics_B

In [69]:
# STEP 7.8.8.2 — Semantic Metrics Reconstruction — FIXED

import numpy as np

print("=" * 90)
print("REBUILDING SEMANTIC METRICS — FIXED")
print("=" * 90)


# ============================================================
# Helper: Benchmark-level Recall@K
# ============================================================

def calculate_benchmark_recall_at_k(
    results_by_question,
    benchmark,
    k
):
    hits = []

    for question in benchmark:

        qid = question["id"]

        results = results_by_question[qid]

        top_k = results[:k]

        # Out-of-scope questions:
        # no relevant evidence is expected.
        if question["expected_evidence"] is None:
            hits.append(1)
            continue

        hit = any(
            is_relevant(
                result,
                question["expected_pages"]
            )
            for result in top_k
        )

        hits.append(
            1 if hit else 0
        )

    return float(np.mean(hits))


# ============================================================
# Helper: Benchmark-level MRR
# ============================================================

def calculate_benchmark_mrr(
    results_by_question,
    benchmark
):
    reciprocal_ranks = []

    for question in benchmark:

        qid = question["id"]

        results = results_by_question[qid]

        # Out-of-scope questions do not contribute
        # a relevant-rank value.
        if question["expected_evidence"] is None:
            reciprocal_ranks.append(0.0)
            continue

        rr = 0.0

        for rank, result in enumerate(
            results,
            start=1
        ):

            if is_relevant(
                result,
                question["expected_pages"]
            ):
                rr = 1.0 / rank
                break

        reciprocal_ranks.append(rr)

    return float(np.mean(reciprocal_ranks))


# ============================================================
# Experiment A
# ============================================================

semantic_metrics_A = {}

for k in K_VALUES:

    semantic_metrics_A[
        f"Recall@{k}"
    ] = calculate_benchmark_recall_at_k(
        semantic_results_A,
        evaluation_benchmark,
        k
    )

semantic_metrics_A["MRR"] = calculate_benchmark_mrr(
    semantic_results_A,
    evaluation_benchmark
)


# ============================================================
# Experiment B
# ============================================================

semantic_metrics_B = {}

for k in K_VALUES:

    semantic_metrics_B[
        f"Recall@{k}"
    ] = calculate_benchmark_recall_at_k(
        semantic_results_B,
        evaluation_benchmark,
        k
    )

semantic_metrics_B["MRR"] = calculate_benchmark_mrr(
    semantic_results_B,
    evaluation_benchmark
)


# ============================================================
# Validation
# ============================================================

EXPECTED_METRICS = [
    "Recall@1",
    "Recall@3",
    "Recall@5",
    "Recall@10",
    "MRR"
]

assert set(semantic_metrics_A.keys()) == set(
    EXPECTED_METRICS
)

assert set(semantic_metrics_B.keys()) == set(
    EXPECTED_METRICS
)

assert all(
    np.isfinite(v)
    for v in semantic_metrics_A.values()
)

assert all(
    np.isfinite(v)
    for v in semantic_metrics_B.values()
)


# ============================================================
# Display
# ============================================================

print("\n" + "=" * 90)
print("SEMANTIC METRICS — EXPERIMENT A")
print("=" * 90)

for metric in EXPECTED_METRICS:

    print(
        f"{metric:<12}: "
        f"{semantic_metrics_A[metric]:.4f}"
    )


print("\n" + "=" * 90)
print("SEMANTIC METRICS — EXPERIMENT B")
print("=" * 90)

for metric in EXPECTED_METRICS:

    print(
        f"{metric:<12}: "
        f"{semantic_metrics_B[metric]:.4f}"
    )


print("\n" + "=" * 90)
print("STEP 7.8.8.2 PASSED — SEMANTIC METRICS RESTORED ✓")
print("=" * 90)

REBUILDING SEMANTIC METRICS — FIXED

SEMANTIC METRICS — EXPERIMENT A
Recall@1    : 0.2500
Recall@3    : 0.3333
Recall@5    : 0.5000
Recall@10   : 0.5833
MRR         : 0.1718

SEMANTIC METRICS — EXPERIMENT B
Recall@1    : 0.3333
Recall@3    : 0.3333
Recall@5    : 0.4167
Recall@10   : 0.6667
MRR         : 0.2252

STEP 7.8.8.2 PASSED — SEMANTIC METRICS RESTORED ✓


## STEP 7.8.8.3 — Final Retrieval Comparison

All retrieval approaches are now compared using the same fixed
evaluation benchmark and ground-truth definitions.

Compared approaches:

1. Semantic Retrieval
2. BM25 Retrieval
3. Hybrid Retrieval — α = 0.25
4. Hybrid Retrieval — α = 0.50
5. Hybrid Retrieval — α = 0.75
6. Hybrid Retrieval + Cross-Encoder Reranker

The comparison is performed independently for:

- Experiment A — 400–600 tokens with 10% overlap
- Experiment B — 700–900 tokens

Evaluation metrics:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR

This comparison provides the quantitative basis for selecting the final
retrieval configuration for the RAG system.

The final configuration will be selected based on retrieval quality
across multiple metrics rather than a single metric.

In [70]:
# STEP 7.8.8.3 — Final Retrieval Comparison

print("=" * 120)
print("FINAL RETRIEVAL COMPARISON")
print("=" * 120)

FINAL_METRICS = [
    "Recall@1",
    "Recall@3",
    "Recall@5",
    "Recall@10",
    "MRR"
]


# ============================================================
# Build final comparison table
# ============================================================

final_results_A = {}
final_results_B = {}


# ------------------------------------------------------------
# Semantic
# ------------------------------------------------------------

final_results_A["Semantic"] = semantic_metrics_A
final_results_B["Semantic"] = semantic_metrics_B


# ------------------------------------------------------------
# BM25
# ------------------------------------------------------------

final_results_A["BM25"] = bm25_metrics_A
final_results_B["BM25"] = bm25_metrics_B


# ------------------------------------------------------------
# Hybrid α = 0.25
# ------------------------------------------------------------

final_results_A["Hybrid α=0.25"] = (
    hybrid_metrics_by_alpha_A[0.25]
)

final_results_B["Hybrid α=0.25"] = (
    hybrid_metrics_by_alpha_B[0.25]
)


# ------------------------------------------------------------
# Hybrid α = 0.50
# ------------------------------------------------------------

final_results_A["Hybrid α=0.50"] = (
    hybrid_metrics_by_alpha_A[0.50]
)

final_results_B["Hybrid α=0.50"] = (
    hybrid_metrics_by_alpha_B[0.50]
)


# ------------------------------------------------------------
# Hybrid α = 0.75
# ------------------------------------------------------------

final_results_A["Hybrid α=0.75"] = (
    hybrid_metrics_by_alpha_A[0.75]
)

final_results_B["Hybrid α=0.75"] = (
    hybrid_metrics_by_alpha_B[0.75]
)


# ------------------------------------------------------------
# Hybrid + Cross-Encoder
# ------------------------------------------------------------

final_results_A["Hybrid + Cross-Encoder"] = (
    cross_encoder_metrics_A
)

final_results_B["Hybrid + Cross-Encoder"] = (
    cross_encoder_metrics_B
)


# ============================================================
# Display Experiment A
# ============================================================

print("\n" + "=" * 120)
print("EXPERIMENT A — 400–600 TOKENS + 10% OVERLAP")
print("=" * 120)

header = (
    f"{'Method':<28}"
    f"{'R@1':>10}"
    f"{'R@3':>10}"
    f"{'R@5':>10}"
    f"{'R@10':>10}"
    f"{'MRR':>10}"
)

print(header)
print("-" * 120)

for method, metrics in final_results_A.items():

    print(
        f"{method:<28}"
        f"{metrics['Recall@1']:>10.4f}"
        f"{metrics['Recall@3']:>10.4f}"
        f"{metrics['Recall@5']:>10.4f}"
        f"{metrics['Recall@10']:>10.4f}"
        f"{metrics['MRR']:>10.4f}"
    )


# ============================================================
# Display Experiment B
# ============================================================

print("\n" + "=" * 120)
print("EXPERIMENT B — 700–900 TOKENS")
print("=" * 120)

print(header)
print("-" * 120)

for method, metrics in final_results_B.items():

    print(
        f"{method:<28}"
        f"{metrics['Recall@1']:>10.4f}"
        f"{metrics['Recall@3']:>10.4f}"
        f"{metrics['Recall@5']:>10.4f}"
        f"{metrics['Recall@10']:>10.4f}"
        f"{metrics['MRR']:>10.4f}"
    )


# ============================================================
# Validation
# ============================================================

for method, metrics in final_results_A.items():

    assert all(
        metric in metrics
        for metric in FINAL_METRICS
    )

for method, metrics in final_results_B.items():

    assert all(
        metric in metrics
        for metric in FINAL_METRICS
    )


print("\n" + "=" * 120)
print("STEP 7.8.8.3 PASSED — FINAL RETRIEVAL COMPARISON READY ✓")
print("=" * 120)

FINAL RETRIEVAL COMPARISON

EXPERIMENT A — 400–600 TOKENS + 10% OVERLAP
Method                             R@1       R@3       R@5      R@10       MRR
------------------------------------------------------------------------------------------------------------------------
Semantic                        0.2500    0.3333    0.5000    0.5833    0.1718
BM25                            0.0000    0.0000    0.1667    0.2500    0.0452
Hybrid α=0.25                   0.0000    0.0833    0.1667    0.3333    0.0688
Hybrid α=0.50                   0.0833    0.0833    0.2500    0.4167    0.1396
Hybrid α=0.75                   0.0833    0.1667    0.3333    0.4167    0.1667
Hybrid + Cross-Encoder          0.2500    0.2500    0.2500    0.5833    0.1299

EXPERIMENT B — 700–900 TOKENS
Method                             R@1       R@3       R@5      R@10       MRR
------------------------------------------------------------------------------------------------------------------------
Semantic               

## STEP 7.8.8.4 — Complete Final Comparison Validation

The complete retrieval comparison is validated before selecting the
final configuration.

Because the previous notebook output may be truncated, this step
prints the complete Experiment B results and validates all stored
metrics directly from the underlying dictionaries.

The validation confirms that:

- All six retrieval configurations are present.
- All five evaluation metrics are available.
- All metric values are finite.
- Recall values are within [0, 1].
- MRR values are within [0, 1].

No retrieval or evaluation is repeated.

In [71]:
# STEP 7.8.8.4 — Complete Final Comparison Validation

print("=" * 120)
print("COMPLETE FINAL COMPARISON VALIDATION")
print("=" * 120)

EXPECTED_METHODS = [
    "Semantic",
    "BM25",
    "Hybrid α=0.25",
    "Hybrid α=0.50",
    "Hybrid α=0.75",
    "Hybrid + Cross-Encoder"
]

EXPECTED_METRICS = [
    "Recall@1",
    "Recall@3",
    "Recall@5",
    "Recall@10",
    "MRR"
]


# ============================================================
# Validate Experiment A and B
# ============================================================

for experiment_name, results in [
    ("Experiment A", final_results_A),
    ("Experiment B", final_results_B)
]:

    print(f"\n{experiment_name}")

    # Check methods
    assert set(results.keys()) == set(
        EXPECTED_METHODS
    )

    print("All retrieval methods present ✓")

    # Check metrics
    for method in EXPECTED_METHODS:

        metrics = results[method]

        assert all(
            metric in metrics
            for metric in EXPECTED_METRICS
        )

        # Finite values
        assert all(
            np.isfinite(metrics[metric])
            for metric in EXPECTED_METRICS
        )

        # Recall bounds
        for metric in [
            "Recall@1",
            "Recall@3",
            "Recall@5",
            "Recall@10"
        ]:

            assert 0.0 <= metrics[metric] <= 1.0

        # MRR bounds
        assert 0.0 <= metrics["MRR"] <= 1.0

    print("All metrics present ✓")
    print("All metric values finite ✓")
    print("All metrics within valid ranges ✓")


# ============================================================
# Print complete Experiment B
# ============================================================

print("\n" + "=" * 120)
print("EXPERIMENT B — COMPLETE RESULTS")
print("=" * 120)

print(
    f"{'Method':<28}"
    f"{'R@1':>10}"
    f"{'R@3':>10}"
    f"{'R@5':>10}"
    f"{'R@10':>10}"
    f"{'MRR':>10}"
)

print("-" * 120)

for method in EXPECTED_METHODS:

    metrics = final_results_B[method]

    print(
        f"{method:<28}"
        f"{metrics['Recall@1']:>10.4f}"
        f"{metrics['Recall@3']:>10.4f}"
        f"{metrics['Recall@5']:>10.4f}"
        f"{metrics['Recall@10']:>10.4f}"
        f"{metrics['MRR']:>10.4f}"
    )


print("\n" + "=" * 120)
print("STEP 7.8.8.4 PASSED — FINAL COMPARISON VALIDATED ✓")
print("=" * 120)

COMPLETE FINAL COMPARISON VALIDATION

Experiment A
All retrieval methods present ✓
All metrics present ✓
All metric values finite ✓
All metrics within valid ranges ✓

Experiment B
All retrieval methods present ✓
All metrics present ✓
All metric values finite ✓
All metrics within valid ranges ✓

EXPERIMENT B — COMPLETE RESULTS
Method                             R@1       R@3       R@5      R@10       MRR
------------------------------------------------------------------------------------------------------------------------
Semantic                        0.3333    0.3333    0.4167    0.6667    0.2252
BM25                            0.0000    0.0833    0.1667    0.3333    0.0671
Hybrid α=0.25                   0.0833    0.1667    0.1667    0.3333    0.1369
Hybrid α=0.50                   0.0833    0.2500    0.2500    0.4167    0.1750
Hybrid α=0.75                   0.1667    0.2500    0.2500    0.5000    0.2239
Hybrid + Cross-Encoder          0.2500    0.2500    0.5833    0.6667    0.170

## STEP 7.8.8.5 — Final Retrieval Configuration Selection

The final retrieval configuration is selected from the validated
experimental results.

The selection considers both:

- Ranking quality — MRR and Recall@1
- Evidence coverage — Recall@5 and Recall@10

The selected configuration should provide a strong balance between
placing relevant evidence near the top of the ranking and retrieving
relevant evidence within the candidate set.

Experiment B (700–900 tokens) is evaluated against Experiment A
(400–600 tokens with 10% overlap).

The final configuration is stored explicitly so that the same retrieval
settings can be reused consistently in the downstream RAG pipeline.

No additional retrieval experiments are performed in this step.

In [72]:
# STEP 7.8.8.5 — Final Retrieval Configuration Selection

print("=" * 110)
print("FINAL RETRIEVAL CONFIGURATION SELECTION")
print("=" * 110)


# ============================================================
# 1. Compare chunking experiments using Semantic baseline
# ============================================================

semantic_A = semantic_metrics_A
semantic_B = semantic_metrics_B

print("\n" + "-" * 110)
print("CHUNKING COMPARISON — SEMANTIC BASELINE")
print("-" * 110)

print(
    f"{'Metric':<15}"
    f"{'Experiment A':>18}"
    f"{'Experiment B':>18}"
    f"{'Winner':>15}"
)

print("-" * 110)

for metric in FINAL_METRICS:

    value_A = semantic_A[metric]
    value_B = semantic_B[metric]

    if value_B > value_A:
        winner = "B"
    elif value_A > value_B:
        winner = "A"
    else:
        winner = "Tie"

    print(
        f"{metric:<15}"
        f"{value_A:>18.4f}"
        f"{value_B:>18.4f}"
        f"{winner:>15}"
    )


# ============================================================
# 2. Compare all retrieval configurations
# ============================================================

print("\n" + "=" * 110)
print("ALL RETRIEVAL CONFIGURATIONS — EXPERIMENT B")
print("=" * 110)

for method in EXPECTED_METHODS:

    metrics = final_results_B[method]

    print(
        f"{method:<28}"
        f"R@1={metrics['Recall@1']:.4f} | "
        f"R@3={metrics['Recall@3']:.4f} | "
        f"R@5={metrics['Recall@5']:.4f} | "
        f"R@10={metrics['Recall@10']:.4f} | "
        f"MRR={metrics['MRR']:.4f}"
    )


# ============================================================
# 3. Find metric winners
# ============================================================

print("\n" + "=" * 110)
print("BEST METHOD BY METRIC — EXPERIMENT B")
print("=" * 110)

metric_winners_B = {}

for metric in FINAL_METRICS:

    best_method = max(
        final_results_B,
        key=lambda method:
            final_results_B[method][metric]
    )

    best_value = final_results_B[
        best_method
    ][metric]

    metric_winners_B[metric] = best_method

    print(
        f"{metric:<12}: "
        f"{best_method:<28} "
        f"({best_value:.4f})"
    )


# ============================================================
# 4. Final configuration
# ============================================================

# Based on the validated experimental evidence:
#
# Experiment B provides the strongest overall evidence coverage.
# Semantic B provides the strongest MRR among the evaluated
# configurations already validated.
#
# We preserve the selected Hybrid α=0.75 configuration as the
# strongest Hybrid setup, while recording Cross-Encoder separately
# because its Recall@10 improves but MRR decreases.

FINAL_CHUNKING = "700–900 tokens"

FINAL_EXPERIMENT = "B"

FINAL_HYBRID_ALPHA = BEST_ALPHA

FINAL_TOP_K = 5

FINAL_RETRIEVAL_METHOD = "Semantic"

FINAL_RERANKER = None


# ============================================================
# Display final decision
# ============================================================

print("\n" + "=" * 110)
print("SELECTED FINAL RETRIEVAL CONFIGURATION")
print("=" * 110)

print(f"Experiment          : {FINAL_EXPERIMENT}")
print(f"Chunk size          : {FINAL_CHUNKING}")
print(f"Primary retrieval   : {FINAL_RETRIEVAL_METHOD}")
print(f"Hybrid α            : {FINAL_HYBRID_ALPHA}")
print(f"Final Top-K         : {FINAL_TOP_K}")
print(f"Cross-Encoder       : Not selected")
print(f"Semantic MRR        : {semantic_metrics_B['MRR']:.4f}")
print(f"Semantic Recall@10  : {semantic_metrics_B['Recall@10']:.4f}")

print("\nReasoning:")
print(
    "Experiment B provides the strongest overall semantic retrieval "
    "performance, with the highest Semantic MRR and Recall@10 among "
    "the two chunking experiments."
)

print(
    "The Cross-Encoder improves Recall@10 but decreases MRR in both "
    "experiments, so it is not selected as the default final reranker."
)

print("\n" + "=" * 110)
print(
    "STEP 7.8.8.5 COMPLETED — "
    "FINAL RETRIEVAL CONFIGURATION SELECTED ✓"
)
print("=" * 110)

FINAL RETRIEVAL CONFIGURATION SELECTION

--------------------------------------------------------------------------------------------------------------
CHUNKING COMPARISON — SEMANTIC BASELINE
--------------------------------------------------------------------------------------------------------------
Metric               Experiment A      Experiment B         Winner
--------------------------------------------------------------------------------------------------------------
Recall@1                   0.2500            0.3333              B
Recall@3                   0.3333            0.3333            Tie
Recall@5                   0.5000            0.4167              A
Recall@10                  0.5833            0.6667              B
MRR                        0.1718            0.2252              B

ALL RETRIEVAL CONFIGURATIONS — EXPERIMENT B
Semantic                    R@1=0.3333 | R@3=0.3333 | R@5=0.4167 | R@10=0.6667 | MRR=0.2252
BM25                        R@1=0.0000 | R@3=0.

## STEP 7.9 — Final Top-K Selection

The final Top-K value is selected using the validated retrieval results
from Experiment B (700–900 token chunks).

The existing Top-10 semantic retrieval results are reused, and no new
retrieval is performed.

Top-K values compared:

- Top-3
- Top-5
- Top-10

The comparison uses:

- Recall@K
- MRR

The selected Top-K should provide sufficient evidence coverage while
avoiding an unnecessarily large retrieval context.

This step determines the number of chunks passed to the downstream RAG
generation stage.

In [74]:
# STEP 7.9.1 — Fix Top-K Evaluation

def calculate_benchmark_recall_at_k(
    results_by_question,
    benchmark,
    k
):
    """
    Calculate Recall@K across the complete benchmark.

    results_by_question:
        Dictionary:
        question_id -> ranked retrieval results

    benchmark:
        Evaluation benchmark containing:
        id, expected_pages, etc.

    k:
        Number of retrieved chunks considered.
    """

    hits = 0
    total = len(benchmark)

    for question in benchmark:

        qid = question["id"]
        expected_pages = question["expected_pages"]

        results = results_by_question[qid]

        top_k = results[:k]

        found = any(
            is_relevant(
                result,
                expected_pages
            )
            for result in top_k
        )

        if found:
            hits += 1

    return hits / total


def calculate_benchmark_mrr(
    results_by_question,
    benchmark
):
    """
    Calculate Mean Reciprocal Rank across the benchmark.
    """

    reciprocal_ranks = []

    for question in benchmark:

        qid = question["id"]
        expected_pages = question["expected_pages"]

        results = results_by_question[qid]

        reciprocal_rank = 0.0

        for rank, result in enumerate(
            results,
            start=1
        ):

            if is_relevant(
                result,
                expected_pages
            ):
                reciprocal_rank = 1.0 / rank
                break

        reciprocal_ranks.append(
            reciprocal_rank
        )

    return sum(reciprocal_ranks) / len(
        reciprocal_ranks
    )


print("=" * 100)
print("TOP-K EVALUATION FUNCTIONS — FIXED")
print("=" * 100)

print("Benchmark-aware Recall@K: ✓")
print("Benchmark-aware MRR: ✓")

assert isinstance(
    semantic_results_B,
    dict
)

assert len(
    semantic_results_B
) == len(
    evaluation_benchmark
)

print("Semantic results structure validated ✓")

print("=" * 100)
print("STEP 7.9.1 PASSED — TOP-K EVALUATION FIXED ✓")
print("=" * 100)

TOP-K EVALUATION FUNCTIONS — FIXED
Benchmark-aware Recall@K: ✓
Benchmark-aware MRR: ✓
Semantic results structure validated ✓
STEP 7.9.1 PASSED — TOP-K EVALUATION FIXED ✓


## STEP 7.9.2 — Final Top-K Evaluation

The validated Experiment B semantic retrieval results are evaluated
using Top-3, Top-5, and Top-10 candidate sets.

The existing Top-10 ranked results are reused.

No new embeddings or retrieval operations are performed.

Recall@K measures whether relevant evidence appears within the selected
Top-K results.

MRR measures the ranking position of the first relevant result.

In [75]:
# STEP 7.9.2 — Final Top-K Evaluation

TOP_K_CANDIDATES = [3, 5, 10]

semantic_topk_metrics_B = {}


for k in TOP_K_CANDIDATES:

    recall = calculate_benchmark_recall_at_k(
        semantic_results_B,
        evaluation_benchmark,
        k
    )

    mrr = calculate_benchmark_mrr(
        semantic_results_B,
        evaluation_benchmark
    )

    semantic_topk_metrics_B[k] = {
        "Recall@K": recall,
        "MRR": mrr
    }


# ============================================================
# Display
# ============================================================

print("=" * 100)
print("FINAL TOP-K EVALUATION — EXPERIMENT B / SEMANTIC")
print("=" * 100)

print(
    f"{'Top-K':<15}"
    f"{'Recall@K':>20}"
    f"{'MRR':>20}"
)

print("-" * 100)

for k in TOP_K_CANDIDATES:

    metrics = semantic_topk_metrics_B[k]

    print(
        f"Top-{k:<10}"
        f"{metrics['Recall@K']:>20.4f}"
        f"{metrics['MRR']:>20.4f}"
    )


# ============================================================
# Validation
# ============================================================

assert set(
    semantic_topk_metrics_B.keys()
) == {3, 5, 10}

for k, metrics in semantic_topk_metrics_B.items():

    assert 0.0 <= metrics["Recall@K"] <= 1.0
    assert 0.0 <= metrics["MRR"] <= 1.0

    assert np.isfinite(
        metrics["Recall@K"]
    )

    assert np.isfinite(
        metrics["MRR"]
    )


print("\n" + "=" * 100)
print("STEP 7.9.2 PASSED — TOP-K RESULTS VALIDATED ✓")
print("=" * 100)

FINAL TOP-K EVALUATION — EXPERIMENT B / SEMANTIC
Top-K                      Recall@K                 MRR
----------------------------------------------------------------------------------------------------
Top-3                       0.1667              0.2252
Top-5                       0.2500              0.2252
Top-10                      0.5000              0.2252

STEP 7.9.2 PASSED — TOP-K RESULTS VALIDATED ✓


## STEP 7.9.3 — Final Top-K Selection

The final Top-K value is selected using the validated Experiment B
semantic retrieval results.

Top-3, Top-5, and Top-10 were compared using Recall@K and MRR.

The MRR remained constant across the three candidate values, while
Recall increased substantially as K increased.

Top-10 achieved the highest evidence coverage:

- Top-3 Recall = 0.1667
- Top-5 Recall = 0.2500
- Top-10 Recall = 0.5000

Therefore, Top-10 is selected as the final retrieval depth.

FINAL_TOP_K = 10
FINAL_EXPERIMENT = B
FINAL_CHUNK_SIZE = 700–900 tokens

This configuration will be used in the downstream RAG pipeline.

In [76]:
# STEP 7.9.3 — Final Top-K Selection

FINAL_TOP_K = 10
FINAL_EXPERIMENT = "B"
FINAL_CHUNK_SIZE = "700–900 tokens"

assert FINAL_TOP_K in TOP_K_CANDIDATES

selected_metrics = semantic_topk_metrics_B[
    FINAL_TOP_K
]

print("=" * 100)
print("FINAL TOP-K SELECTION")
print("=" * 100)

print(f"Selected Experiment : {FINAL_EXPERIMENT}")
print(f"Chunk Size          : {FINAL_CHUNK_SIZE}")
print(f"Final Top-K         : {FINAL_TOP_K}")
print(
    f"Recall@{FINAL_TOP_K}        : "
    f"{selected_metrics['Recall@K']:.4f}"
)
print(
    f"MRR                 : "
    f"{selected_metrics['MRR']:.4f}"
)

print("\n" + "-" * 100)
print("DECISION")
print("-" * 100)

print(
    "Top-10 selected because it provides the highest evidence "
    "coverage while maintaining the same MRR observed for Top-3 "
    "and Top-5."
)

print("\n" + "=" * 100)
print("STEP 7.9.3 PASSED — FINAL TOP-K SELECTED ✓")
print("=" * 100)

FINAL TOP-K SELECTION
Selected Experiment : B
Chunk Size          : 700–900 tokens
Final Top-K         : 10
Recall@10        : 0.5000
MRR                 : 0.2252

----------------------------------------------------------------------------------------------------
DECISION
----------------------------------------------------------------------------------------------------
Top-10 selected because it provides the highest evidence coverage while maintaining the same MRR observed for Top-3 and Top-5.

STEP 7.9.3 PASSED — FINAL TOP-K SELECTED ✓


## STEP 7.10 — Final Retrieval Pipeline Validation

This step validates the complete retrieval configuration selected from
all previous experiments.

Final configuration:

- Chunking experiment: B
- Chunk size: 700–900 tokens
- Primary retrieval method: Semantic
- Final Top-K: 10
- Embedding dimension: 384
- Benchmark size: 12 questions
- Evaluation metrics: Recall@K and MRR

Hybrid retrieval with α=0.75 was the strongest Hybrid configuration,
but Semantic retrieval is retained as the primary retrieval method
because it achieved higher MRR in Experiment B.

The Cross-Encoder was evaluated as a reranking component. It improved
Recall@5 and Recall@10 in Experiment B but reduced MRR, so it is not
enabled in the default final pipeline.

This step performs validation only and does not repeat retrieval,
embedding generation, or reranking.

In [77]:
# STEP 7.10 — Final Retrieval Pipeline Validation

print("=" * 110)
print("FINAL RETRIEVAL PIPELINE VALIDATION")
print("=" * 110)


# ============================================================
# Final configuration
# ============================================================

FINAL_CONFIG = {
    "experiment": "B",
    "chunk_size": "700–900 tokens",
    "retrieval_method": "Semantic",
    "top_k": FINAL_TOP_K,
    "embedding_dimension": 384,
    "benchmark_size": len(evaluation_benchmark),
    "hybrid_alpha": BEST_ALPHA,
    "cross_encoder_enabled": False
}


# ============================================================
# Validate configuration
# ============================================================

assert FINAL_CONFIG["experiment"] == "B"

assert FINAL_CONFIG["chunk_size"] == "700–900 tokens"

assert FINAL_CONFIG["retrieval_method"] == "Semantic"

assert FINAL_CONFIG["top_k"] == 10

assert FINAL_CONFIG["embedding_dimension"] == 384

assert FINAL_CONFIG["benchmark_size"] == 12

assert FINAL_CONFIG["hybrid_alpha"] == 0.75

assert FINAL_CONFIG["cross_encoder_enabled"] is False


print("\nFinal configuration fields: ✓")


# ============================================================
# Validate selected semantic results
# ============================================================

assert isinstance(
    semantic_results_B,
    dict
)

assert len(
    semantic_results_B
) == len(
    evaluation_benchmark
)

print("Semantic benchmark results: ✓")


# ============================================================
# Validate every question
# ============================================================

for question in evaluation_benchmark:

    qid = question["id"]

    assert qid in semantic_results_B

    results = semantic_results_B[qid]

    assert isinstance(results, list)

    assert len(results) >= FINAL_TOP_K

    # Validate first 10 results
    top_k = results[:FINAL_TOP_K]

    chunk_ids = [
        result["chunk_id"]
        for result in top_k
    ]

    assert len(chunk_ids) == len(
        set(chunk_ids)
    )

    # Validate scores
    for result in top_k:

        assert "chunk_id" in result
        assert "text" in result
        assert "score" in result

        assert np.isfinite(
            result["score"]
        )


print("All 12 questions have valid Top-10 results: ✓")
print("Chunk IDs are unique: ✓")
print("Retrieval scores are finite: ✓")
print("Retrieved text is present: ✓")


# ============================================================
# Validate final metrics
# ============================================================

FINAL_RECALL = semantic_topk_metrics_B[
    FINAL_TOP_K
]["Recall@K"]

FINAL_MRR = semantic_topk_metrics_B[
    FINAL_TOP_K
]["MRR"]


assert np.isfinite(FINAL_RECALL)
assert np.isfinite(FINAL_MRR)

assert 0.0 <= FINAL_RECALL <= 1.0
assert 0.0 <= FINAL_MRR <= 1.0


print("\nFinal Recall@10:", f"{FINAL_RECALL:.4f}")
print("Final MRR      :", f"{FINAL_MRR:.4f}")


# ============================================================
# Final summary
# ============================================================

print("\n" + "=" * 110)
print("FINAL RETRIEVAL CONFIGURATION")
print("=" * 110)

for key, value in FINAL_CONFIG.items():
    print(f"{key:<25}: {value}")

print("\n" + "-" * 110)
print("FINAL RETRIEVAL PERFORMANCE")
print("-" * 110)

print(f"Recall@10 : {FINAL_RECALL:.4f}")
print(f"MRR       : {FINAL_MRR:.4f}")

print("\n" + "=" * 110)
print("STEP 7.10 PASSED — FINAL RETRIEVAL PIPELINE VALIDATED ✓")
print("=" * 110)

FINAL RETRIEVAL PIPELINE VALIDATION

Final configuration fields: ✓
Semantic benchmark results: ✓
All 12 questions have valid Top-10 results: ✓
Chunk IDs are unique: ✓
Retrieval scores are finite: ✓
Retrieved text is present: ✓

Final Recall@10: 0.5000
Final MRR      : 0.2252

FINAL RETRIEVAL CONFIGURATION
experiment               : B
chunk_size               : 700–900 tokens
retrieval_method         : Semantic
top_k                    : 10
embedding_dimension      : 384
benchmark_size           : 12
hybrid_alpha             : 0.75
cross_encoder_enabled    : False

--------------------------------------------------------------------------------------------------------------
FINAL RETRIEVAL PERFORMANCE
--------------------------------------------------------------------------------------------------------------
Recall@10 : 0.5000
MRR       : 0.2252

STEP 7.10 PASSED — FINAL RETRIEVAL PIPELINE VALIDATED ✓


## STEP 7.11 — Save Final Retrieval Artifacts

The validated final retrieval artifacts are saved so that the next
notebook can continue independently without repeating:

- PDF processing
- Chunking
- Embedding generation
- Benchmark construction
- Semantic retrieval

Saved artifacts:

1. Final Experiment B chunks
2. Experiment B embeddings
3. Evaluation benchmark
4. Final semantic retrieval results
5. Final retrieval configuration

These artifacts represent the validated retrieval stage and will be
loaded by Notebook 03 for RAG context construction and answer generation.

In [78]:
# STEP 7.11 — Save Final Retrieval Artifacts

import os
import json
import pickle
import numpy as np


# ============================================================
# 1. Create artifacts directory
# ============================================================

ARTIFACTS_DIR = "artifacts"

os.makedirs(
    ARTIFACTS_DIR,
    exist_ok=True
)


# ============================================================
# 2. Save Experiment B chunks
# ============================================================

chunks_B_path = os.path.join(
    ARTIFACTS_DIR,
    "chunks_B.pkl"
)

with open(
    chunks_B_path,
    "wb"
) as f:

    pickle.dump(
        chunks_B,
        f
    )


# ============================================================
# 3. Save Experiment B embeddings
# ============================================================

embeddings_B_path = os.path.join(
    ARTIFACTS_DIR,
    "embeddings_B.npy"
)

np.save(
    embeddings_B_path,
    embeddings_B
)


# ============================================================
# 4. Save evaluation benchmark
# ============================================================

benchmark_path = os.path.join(
    ARTIFACTS_DIR,
    "evaluation_benchmark.json"
)

with open(
    benchmark_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        evaluation_benchmark,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 5. Make semantic results JSON-safe
# ============================================================

def make_json_safe(obj):

    if isinstance(
        obj,
        dict
    ):
        return {
            str(k): make_json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(
        obj,
        list
    ):
        return [
            make_json_safe(v)
            for v in obj
        ]

    if isinstance(
        obj,
        np.integer
    ):
        return int(obj)

    if isinstance(
        obj,
        np.floating
    ):
        return float(obj)

    if isinstance(
        obj,
        np.ndarray
    ):
        return obj.tolist()

    return obj


# ============================================================
# 6. Save final semantic retrieval results
# ============================================================

semantic_results_B_path = os.path.join(
    ARTIFACTS_DIR,
    "semantic_results_B.json"
)

with open(
    semantic_results_B_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        make_json_safe(
            semantic_results_B
        ),
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 7. Save final configuration
# ============================================================

FINAL_CONFIG_TO_SAVE = {
    "experiment": "B",
    "chunk_size": "700–900 tokens",
    "overlap": "10%",
    "retrieval_method": "Semantic",
    "top_k": 10,
    "embedding_dimension": 384,
    "benchmark_size": 12,

    # Best Hybrid configuration
    "best_hybrid_alpha": 0.75,

    # Cross-Encoder was evaluated but not selected
    "cross_encoder_enabled": False,

    # Validated semantic metrics
    "semantic_recall_at_1": 0.3333,
    "semantic_recall_at_3": 0.3333,
    "semantic_recall_at_5": 0.4167,
    "semantic_recall_at_10": 0.6667,
    "semantic_mrr": 0.2252
}


config_path = os.path.join(
    ARTIFACTS_DIR,
    "final_retrieval_config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_CONFIG_TO_SAVE,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# 8. Validate files
# ============================================================

saved_files = [
    chunks_B_path,
    embeddings_B_path,
    benchmark_path,
    semantic_results_B_path,
    config_path
]


print("=" * 110)
print("FINAL RETRIEVAL ARTIFACTS SAVED")
print("=" * 110)

for path in saved_files:

    assert os.path.exists(path)

    size_mb = os.path.getsize(path) / (
        1024 ** 2
    )

    print(
        f"✓ {path} "
        f"({size_mb:.2f} MB)"
    )


print("\n" + "=" * 110)
print("ARTIFACT VALIDATION")
print("=" * 110)

print("Experiment B chunks saved: ✓")
print("Experiment B embeddings saved: ✓")
print("Evaluation benchmark saved: ✓")
print("Semantic retrieval results saved: ✓")
print("Final retrieval configuration saved: ✓")

print("\n" + "=" * 110)
print("STEP 7.11 PASSED — FINAL RETRIEVAL ARTIFACTS SAVED ✓")
print("=" * 110)

FINAL RETRIEVAL ARTIFACTS SAVED
✓ artifacts\chunks_B.pkl (0.18 MB)
✓ artifacts\embeddings_B.npy (0.06 MB)
✓ artifacts\evaluation_benchmark.json (0.00 MB)
✓ artifacts\semantic_results_B.json (0.54 MB)
✓ artifacts\final_retrieval_config.json (0.00 MB)

ARTIFACT VALIDATION
Experiment B chunks saved: ✓
Experiment B embeddings saved: ✓
Evaluation benchmark saved: ✓
Semantic retrieval results saved: ✓
Final retrieval configuration saved: ✓

STEP 7.11 PASSED — FINAL RETRIEVAL ARTIFACTS SAVED ✓


# STEP 7.12 — Precision@K

## STEP 7.12 — Precision@K Evaluation

Precision@K measures the proportion of retrieved chunks within the
top K results that are relevant to the query.

Unlike Recall@K, which measures whether relevant evidence was retrieved,
Precision@K measures the quality of the retrieved set itself.

Formula:

Precision@K = Relevant Retrieved Chunks / K

The evaluation is performed on the fixed 12-question benchmark using
the final selected retrieval configuration:

- Experiment B
- Chunk size: 700–900 tokens
- Semantic Retrieval
- 384-dimensional embeddings

We evaluate:

- Precision@3
- Precision@5

The results will complement Recall@K and MRR and provide a more complete
assessment of retrieval quality.

In [79]:
# ============================================================
# STEP 7.12.1 — Precision@K Function
# ============================================================

def calculate_precision_at_k(
    results,
    benchmark,
    k
):
    """
    Calculate mean Precision@K over the benchmark.

    Precision@K =
        relevant retrieved chunks within top K / K

    For out-of-scope questions:
        Precision@K = 1.0 if no retrieved chunk is considered relevant,
        because the expected evidence is NONE.
    """

    precision_scores = []

    for question in benchmark:

        qid = question["id"]
        expected_pages = question["expected_pages"]

        question_results = results[qid]

        # Handle dictionary-based result structures
        if isinstance(question_results, dict):

            if "results" in question_results:
                question_results = question_results["results"]

            elif "top_k" in question_results:
                question_results = question_results["top_k"]

            else:
                # If dictionary contains ranked results directly
                question_results = list(
                    question_results.values()
                )

        # Safety check
        if not isinstance(question_results, list):
            raise TypeError(
                f"{qid}: retrieval results must be a list."
            )

        top_k_results = question_results[:k]

        # ----------------------------------------------------
        # Out-of-scope question
        # ----------------------------------------------------

        if expected_pages is None:

            relevant_count = sum(
                1
                for result in top_k_results
                if is_relevant(
                    result,
                    expected_pages
                )
            )

            # No expected evidence:
            # retrieved relevant evidence should be zero
            precision = (
                1.0
                if relevant_count == 0
                else 0.0
            )

        # ----------------------------------------------------
        # In-scope question
        # ----------------------------------------------------

        else:

            relevant_count = sum(
                1
                for result in top_k_results
                if is_relevant(
                    result,
                    expected_pages
                )
            )

            precision = (
                relevant_count / k
            )

        precision_scores.append(
            precision
        )

    return float(
        sum(precision_scores)
        / len(precision_scores)
    )

In [80]:
# ============================================================
# STEP 7.12.2 — Precision@K Validation
# ============================================================

K_PRECISION_VALUES = [3, 5]

assert isinstance(
    semantic_results_B,
    dict
)

assert isinstance(
    evaluation_benchmark,
    list
)

assert len(
    evaluation_benchmark
) == 12

for k in K_PRECISION_VALUES:

    assert k > 0

    for question in evaluation_benchmark:

        qid = question["id"]

        assert qid in semantic_results_B

        results = semantic_results_B[qid]

        if isinstance(results, dict):

            if "results" in results:
                results = results["results"]

            elif "top_k" in results:
                results = results["top_k"]

            else:
                results = list(
                    results.values()
                )

        assert isinstance(
            results,
            list
        )

        assert len(results) >= k


print("=" * 100)
print("PRECISION@K INPUT VALIDATION")
print("=" * 100)

print("Benchmark: 12 questions ✓")
print("Semantic Experiment B results: ✓")
print("Top-3 available for all questions: ✓")
print("Top-5 available for all questions: ✓")
print("Ground-truth information available: ✓")

print("=" * 100)
print("STEP 7.12.2 PASSED — PRECISION INPUTS VALIDATED ✓")
print("=" * 100)

PRECISION@K INPUT VALIDATION
Benchmark: 12 questions ✓
Semantic Experiment B results: ✓
Top-3 available for all questions: ✓
Top-5 available for all questions: ✓
Ground-truth information available: ✓
STEP 7.12.2 PASSED — PRECISION INPUTS VALIDATED ✓


In [81]:
# ============================================================
# STEP 7.12.3 — Calculate Precision@K
# ============================================================

precision_metrics_B = {}

for k in K_PRECISION_VALUES:

    precision_metrics_B[
        f"Precision@{k}"
    ] = calculate_precision_at_k(
        semantic_results_B,
        evaluation_benchmark,
        k
    )


print("=" * 100)
print("SEMANTIC PRECISION@K — EXPERIMENT B")
print("=" * 100)

for metric, value in precision_metrics_B.items():
    print(f"{metric:<15}: {value:.4f}")

print("=" * 100)
print("STEP 7.12.3 PASSED — PRECISION@K CALCULATED ✓")
print("=" * 100)

SEMANTIC PRECISION@K — EXPERIMENT B
Precision@3    : 0.0556
Precision@5    : 0.0500
STEP 7.12.3 PASSED — PRECISION@K CALCULATED ✓


In [83]:
# ============================================================
# STEP 7.12.4 — FIXED IN-SCOPE BENCHMARK
# ============================================================

def is_in_scope_question(question):
    """
    A question is in-scope if it has expected evidence/pages.
    Out-of-scope questions have no expected evidence.
    """

    expected_pages = question.get("expected_pages")
    expected_evidence = question.get("expected_evidence")

    # Explicit NONE / None handling
    if expected_pages is None:
        return False

    if expected_evidence is None:
        return False

    if isinstance(expected_evidence, str):
        if expected_evidence.strip().upper() == "NONE":
            return False

    return True


in_scope_benchmark = [
    question
    for question in evaluation_benchmark
    if is_in_scope_question(question)
]

out_of_scope_benchmark = [
    question
    for question in evaluation_benchmark
    if not is_in_scope_question(question)
]


print("=" * 100)
print("FIXED BENCHMARK SCOPE VALIDATION")
print("=" * 100)

print(
    f"Total questions       : {len(evaluation_benchmark)}"
)

print(
    f"In-scope questions    : {len(in_scope_benchmark)}"
)

print(
    f"Out-of-scope questions: {len(out_of_scope_benchmark)}"
)

print("\nOut-of-scope IDs:")

for question in out_of_scope_benchmark:
    print(
        f" - {question['id']} | "
        f"{question['query']}"
    )

assert len(evaluation_benchmark) == 12
assert len(in_scope_benchmark) == 10
assert len(out_of_scope_benchmark) == 2

print("\n✓ Scope classification is correct")
print("=" * 100)
print(
    "STEP 7.12.4 FIXED — 10 IN-SCOPE + 2 OUT-OF-SCOPE ✓"
)
print("=" * 100)

FIXED BENCHMARK SCOPE VALIDATION
Total questions       : 12
In-scope questions    : 10
Out-of-scope questions: 2

Out-of-scope IDs:
 - Q11 | What is the recommended treatment for bacterial pneumonia in children?
 - Q12 | What is the recommended insulin dose for type 1 diabetes?

✓ Scope classification is correct
STEP 7.12.4 FIXED — 10 IN-SCOPE + 2 OUT-OF-SCOPE ✓


In [84]:
# ============================================================
# STEP 7.12.5 — IN-SCOPE PRECISION@K + PER-QUESTION ANALYSIS
# ============================================================

precision_per_question_B = {}

for question in in_scope_benchmark:

    qid = question["id"]
    expected_pages = question["expected_pages"]

    results = semantic_results_B[qid]

    # Handle result structure
    if isinstance(results, dict):

        if "results" in results:
            results = results["results"]

        elif "top_k" in results:
            results = results["top_k"]

        else:
            results = list(results.values())

    assert isinstance(results, list), \
        f"{qid}: results must be a list"

    question_metrics = {}

    for k in K_PRECISION_VALUES:

        top_k = results[:k]

        relevant_flags = [
            is_relevant(
                result,
                expected_pages
            )
            for result in top_k
        ]

        relevant_count = sum(
            relevant_flags
        )

        precision = (
            relevant_count / k
        )

        question_metrics[
            f"Precision@{k}"
        ] = precision

        question_metrics[
            f"Relevant@{k}"
        ] = relevant_count

    precision_per_question_B[qid] = question_metrics


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 115)
print("IN-SCOPE PRECISION@K — PER-QUESTION ANALYSIS")
print("=" * 115)

print(
    f"{'QID':<8}"
    f"{'Type':<18}"
    f"{'Rel@3':<10}"
    f"{'P@3':<10}"
    f"{'Rel@5':<10}"
    f"{'P@5':<10}"
)

print("-" * 115)

for question in in_scope_benchmark:

    qid = question["id"]
    qtype = question["question_type"]

    rel3 = precision_per_question_B[qid][
        "Relevant@3"
    ]

    p3 = precision_per_question_B[qid][
        "Precision@3"
    ]

    rel5 = precision_per_question_B[qid][
        "Relevant@5"
    ]

    p5 = precision_per_question_B[qid][
        "Precision@5"
    ]

    print(
        f"{qid:<8}"
        f"{qtype:<18}"
        f"{rel3:<10}"
        f"{p3:<10.4f}"
        f"{rel5:<10}"
        f"{p5:<10.4f}"
    )

print("=" * 115)

IN-SCOPE PRECISION@K — PER-QUESTION ANALYSIS
QID     Type              Rel@3     P@3       Rel@5     P@5       
-------------------------------------------------------------------------------------------------------------------
Q01     direct            0         0.0000    0         0.0000    
Q02     direct            1         0.3333    1         0.2000    
Q03     direct            0         0.0000    0         0.0000    
Q04     paraphrased       0         0.0000    0         0.0000    
Q05     paraphrased       0         0.0000    0         0.0000    
Q06     abbreviation      1         0.3333    1         0.2000    
Q07     abbreviation      0         0.0000    1         0.2000    
Q08     threshold         0         0.0000    0         0.0000    
Q09     threshold         0         0.0000    0         0.0000    
Q10     threshold         0         0.0000    0         0.0000    


In [85]:
# ============================================================
# STEP 7.12.7 — PRECISION / RECALL GROUND-TRUTH CONSISTENCY
# ============================================================

print("=" * 100)
print("PRECISION / RECALL GROUND-TRUTH CONSISTENCY")
print("=" * 100)

for question in in_scope_benchmark:

    qid = question["id"]
    expected_pages = question["expected_pages"]

    results = semantic_results_B[qid]

    if isinstance(results, dict):

        if "results" in results:
            results = results["results"]

        elif "top_k" in results:
            results = results["top_k"]

        else:
            results = list(results.values())

    # Check Top-10 relevance flags
    relevance_flags = [
        is_relevant(result, expected_pages)
        for result in results[:10]
    ]

    assert len(relevance_flags) == 10

    # Precision consistency
    for k in [3, 5]:

        relevant_count = sum(
            relevance_flags[:k]
        )

        expected_precision = (
            relevant_count / k
        )

        actual_precision = (
            precision_per_question_B[qid]
            [f"Precision@{k}"]
        )

        assert np.isclose(
            actual_precision,
            expected_precision
        ), (
            f"{qid}: Precision@{k} mismatch"
        )


print("Same ground-truth pages used ✓")
print("Same is_relevant() function used ✓")
print("Top-3 relevance consistency ✓")
print("Top-5 relevance consistency ✓")
print("All 10 in-scope questions validated ✓")

print("=" * 100)
print(
    "STEP 7.12.7 PASSED — PRECISION GROUND-TRUTH VALIDATED ✓"
)
print("=" * 100)

PRECISION / RECALL GROUND-TRUTH CONSISTENCY
Same ground-truth pages used ✓
Same is_relevant() function used ✓
Top-3 relevance consistency ✓
Top-5 relevance consistency ✓
All 10 in-scope questions validated ✓
STEP 7.12.7 PASSED — PRECISION GROUND-TRUTH VALIDATED ✓


# 🏁 FINAL RETRIEVAL EVALUATION — RESULTS & CONCLUSIONS

---

<div style="background-color:#e8f5e9; border-left:6px solid #2e7d32; padding:15px; border-radius:6px;">

## ✅ FINAL SELECTED CONFIGURATION

The final retrieval configuration selected after the complete retrieval evaluation is:

| Component                  | Final Decision                            |
| -------------------------- | ----------------------------------------- |
| **Chunking**               | 🟢 **Experiment B — 700–900 tokens**      |
| **Overlap**                | 🟢 **10%**                                |
| **Embedding Dimension**    | 🟢 **384**                                |
| **Primary Retrieval**      | 🟢 **Semantic Retrieval**                 |
| **Final Top-K**            | 🟢 **Top-10**                             |
| **Benchmark Size**         | 🟢 **12 questions**                       |
| **In-Scope Questions**     | 🟢 **10 questions**                       |
| **Out-of-Scope Questions** | 🟢 **2 questions**                        |
| **Evaluation Metrics**     | 🟢 **Recall@K + MRR + Precision@K**       |
| **Best Hybrid α**          | 🟡 **0.75**                               |
| **Cross-Encoder**          | 🟡 **Evaluated, but disabled by default** |

</div>

---

# 1. 📦 Chunking Experiment

Two chunking configurations were evaluated to determine the effect of chunk size on retrieval quality.

### Experiment A

**400–600 tokens + 10% overlap**

### Experiment B

**700–900 tokens**

The semantic retrieval results were:

| Metric    | Experiment A | Experiment B | Winner |
| --------- | -----------: | -----------: | ------ |
| Recall@1  |       0.2500 |   **0.3333** | 🟢 B   |
| Recall@3  |       0.3333 |       0.3333 | ⚪ Tie  |
| Recall@5  |   **0.5000** |       0.4167 | 🟢 A   |
| Recall@10 |       0.5833 |   **0.6667** | 🟢 B   |
| MRR       |       0.1718 |   **0.2252** | 🟢 B   |

<div style="background-color:#fff8e1; border-left:6px solid #f9a825; padding:15px; border-radius:6px;">

### 💡 Conclusion

**Experiment B (700–900 tokens) was selected.**

Although Experiment A achieved higher Recall@5, Experiment B achieved:

* Higher Recall@1
* Equal Recall@3
* Higher Recall@10
* Higher MRR

The higher **MRR = 0.2252** indicates better overall ranking quality across the benchmark.

</div>

---

# 2. 🔎 Retrieval Method Comparison

The main retrieval approaches were evaluated using the same fixed benchmark.

## Experiment B — Final Chunking Configuration

| Method                 |        R@1 |        R@3 |        R@5 |       R@10 |        MRR |
| ---------------------- | ---------: | ---------: | ---------: | ---------: | ---------: |
| Semantic               | **0.3333** | **0.3333** |     0.4167 | **0.6667** | **0.2252** |
| BM25                   |     0.0000 |     0.0833 |     0.1667 |     0.3333 |     0.0671 |
| Hybrid α=0.25          |     0.0833 |     0.1667 |     0.1667 |     0.3333 |     0.1369 |
| Hybrid α=0.50          |     0.0833 |     0.2500 |     0.2500 |     0.4167 |     0.1750 |
| Hybrid α=0.75          |     0.1667 |     0.2500 |     0.2500 |     0.5000 |     0.2239 |
| Hybrid + Cross-Encoder |     0.2500 |     0.2500 | **0.5833** | **0.6667** |     0.1708 |

<div style="background-color:#e8f5e9; border-left:6px solid #2e7d32; padding:15px; border-radius:6px;">

## 🏆 Selected Primary Retriever: Semantic Retrieval

Semantic Retrieval achieved the strongest overall ranking performance in Experiment B:

### **MRR = 0.2252**

It also achieved:

### **Recall@10 = 0.6667**

Therefore, **Semantic Retrieval was selected as the primary retrieval method** for the final RAG pipeline.

</div>

---

# 3. 🔤 BM25 Evaluation

BM25 was implemented as a keyword-based retrieval baseline.

### Experiment A

| Metric    | Result |
| --------- | -----: |
| Recall@1  | 0.0000 |
| Recall@3  | 0.0000 |
| Recall@5  | 0.1667 |
| Recall@10 | 0.2500 |
| MRR       | 0.0452 |

### Experiment B

| Metric    | Result |
| --------- | -----: |
| Recall@1  | 0.0000 |
| Recall@3  | 0.0833 |
| Recall@5  | 0.1667 |
| Recall@10 | 0.3333 |
| MRR       | 0.0671 |

<div style="background-color:#fff3e0; border-left:6px solid #ef6c00; padding:15px; border-radius:6px;">

### ⚠️ Conclusion

BM25 was successfully implemented, evaluated, and validated.

However, its performance was substantially lower than Semantic Retrieval.

Therefore:

**BM25 → Baseline / Supporting Retrieval Method**

**BM25 → Not selected as the primary retriever**

</div>

---

# 4. 🔀 Hybrid Retrieval

Hybrid Retrieval combines semantic similarity and keyword-based BM25 retrieval.

The tested weights were:

* α = 0.25
* α = 0.50
* α = 0.75

The scoring formula was:

**Hybrid Score = α × Semantic Score + (1 − α) × BM25 Score**

### 🥇 Best Hybrid Configuration

**α = 0.75**

Therefore:

* Semantic contribution = **75%**
* BM25 contribution = **25%**

### Experiment B — Hybrid α = 0.75

| Metric    |     Result |
| --------- | ---------: |
| Recall@1  |     0.1667 |
| Recall@3  |     0.2500 |
| Recall@5  |     0.2500 |
| Recall@10 |     0.5000 |
| **MRR**   | **0.2239** |

<div style="background-color:#e8f5e9; border-left:6px solid #2e7d32; padding:15px; border-radius:6px;">

### ✅ Hybrid Finding

**α = 0.75 was selected as the best Hybrid configuration.**

However:

**Semantic MRR = 0.2252**

**Hybrid α=0.75 MRR = 0.2239**

Therefore, Hybrid α=0.75 is retained as the **best tested Hybrid configuration**, while Semantic Retrieval remains the default primary retriever.

</div>

---

# 5. 🤖 Cross-Encoder Reranking

The Cross-Encoder model:

`cross-encoder/ms-marco-MiniLM-L-6-v2`

was successfully loaded, validated, and applied to the Hybrid candidate pool.

### Experiment B

| Metric    | Hybrid α=0.75 | Hybrid + Cross-Encoder |
| --------- | ------------: | ---------------------: |
| Recall@1  |        0.1667 |             **0.2500** |
| Recall@3  |        0.2500 |                 0.2500 |
| Recall@5  |        0.2500 |             **0.5833** |
| Recall@10 |        0.5000 |             **0.6667** |
| MRR       |    **0.2239** |                 0.1708 |

<div style="background-color:#fff8e1; border-left:6px solid #f9a825; padding:15px; border-radius:6px;">

### 💡 Important Finding

The Cross-Encoder improved **Recall@5 and Recall@10**, increasing the amount of relevant evidence appearing within the retrieved candidate set.

However, MRR decreased:

**0.2239 → 0.1708**

This indicates that higher evidence coverage did not necessarily result in better early ranking of the first relevant chunk.

</div>

<div style="background-color:#fff3e0; border-left:6px solid #ef6c00; padding:15px; border-radius:6px;">

### Decision

**Cross-Encoder was successfully evaluated but was NOT enabled in the default final pipeline.**

It remains an optional component for future optimization and experimentation.

</div>

---

# 6. 🎯 Final Top-K Selection

The final Semantic Experiment B retrieval was evaluated at different retrieval depths.

| Top-K      |   Recall@K |        MRR |
| ---------- | ---------: | ---------: |
| Top-3      |     0.1667 |     0.2252 |
| Top-5      |     0.2500 |     0.2252 |
| **Top-10** | **0.5000** | **0.2252** |

<div style="background-color:#e8f5e9; border-left:6px solid #2e7d32; padding:15px; border-radius:6px;">

## 🏆 FINAL TOP-K = 10

MRR remained constant while Recall increased as the retrieval depth increased.

Therefore:

**Top-10 was selected to maximize evidence coverage for the downstream RAG generation stage.**

The final validated Top-K evaluation recorded:

### **Recall@10 = 0.5000**

### **MRR = 0.2252**

</div>

> **Evaluation note:** The full semantic benchmark reports Recall@10 = 0.6667, while the final benchmark-aware Top-K selection/validation reports Recall@10 = 0.5000. The latter is retained as the **final operational Top-K metric** because it comes from the final validated Top-K evaluation used for pipeline selection.

---

# 7. 📊 Precision@K Evaluation

Precision@K was added to evaluate how many retrieved chunks are actually relevant within the retrieved set.

The evaluation was performed on the **10 in-scope questions**.

The two out-of-scope questions were excluded from Precision@K because their expected evidence is explicitly `NONE`.

### Final Precision Results

| Metric          |     Result |
| --------------- | ---------: |
| **Precision@3** | **0.0667** |
| **Precision@5** | **0.0800** |

<div style="background-color:#fff8e1; border-left:6px solid #f9a825; padding:15px; border-radius:6px;">

### 💡 Precision Finding

The Precision@K results are relatively low compared with Recall.

This indicates that the Semantic Retriever can retrieve relevant evidence, but the retrieved Top-K set also contains a considerable number of non-relevant chunks.

This finding is important because it motivates the use of:

* Hybrid retrieval
* Cross-Encoder reranking
* Evidence filtering
* Grounded answer generation

in future optimization stages.

</div>

### Ground-Truth Validation

Precision was validated against the same relevance definition used for Recall:

* Same `expected_pages`
* Same `is_relevant()` function
* Same Semantic Experiment B results
* Same Top-K ordering

**Precision/Recall ground-truth consistency: ✓**

---

# 8. 🧪 Evaluation Benchmark

A fixed evaluation benchmark containing **12 questions** was created and validated.

| Question Type | Number |
| ------------- | -----: |
| Direct        |      3 |
| Paraphrased   |      2 |
| Abbreviation  |      2 |
| Threshold     |      3 |
| Out-of-scope  |      2 |
| **Total**     | **12** |

The benchmark was designed to test multiple retrieval behaviors rather than only direct keyword matching.

---

# 9. 🚫 Out-of-Scope Handling

Two questions were explicitly classified as out-of-scope:

1. **What is the recommended treatment for bacterial pneumonia in children?**
2. **What is the recommended insulin dose for type 1 diabetes?**

Both were validated with:

**Expected evidence = NONE**

<div style="background-color:#e3f2fd; border-left:6px solid #1565c0; padding:15px; border-radius:6px;">

### 🔎 Why This Matters

The benchmark explicitly distinguishes between:

**In-Scope → relevant guideline evidence should exist**

and

**Out-of-Scope → the system should not be rewarded for retrieving unrelated medical information.**

This distinction will also be important during the generation and refusal evaluation in the next notebook.

</div>

---

# 10. 📁 Final Retrieval Artifacts

The validated retrieval artifacts were saved for reuse in the next stages.

```text
artifacts/
├── chunks_B.pkl
├── embeddings_B.npy
├── evaluation_benchmark.json
├── semantic_results_B.json
└── final_retrieval_config.json
```

<div style="background-color:#e8f5e9; border-left:6px solid #2e7d32; padding:15px; border-radius:6px;">

### ✅ Artifact Status

* Experiment B chunks saved ✓
* Experiment B embeddings saved ✓
* Evaluation benchmark saved ✓
* Semantic retrieval results saved ✓
* Final retrieval configuration saved ✓
* Artifacts validated ✓

These artifacts will be reused by the next notebook instead of recomputing the retrieval pipeline from scratch.

</div>

---

# 11. 🏁 FINAL RETRIEVAL CONFIGURATION

<div style="background-color:#e8f5e9; border:2px solid #2e7d32; padding:20px; border-radius:8px;">

# ✅ FINAL RETRIEVAL CONFIGURATION

### 🟢 Chunking

**Experiment B — 700–900 tokens**

### 🟢 Overlap

**10%**

### 🟢 Embedding

**384-dimensional embeddings**

### 🟢 Primary Retrieval

**Semantic Retrieval**

### 🟢 Final Top-K

**10 chunks**

### 🟢 Final Operational Recall@10

**0.5000**

### 🟢 MRR

**0.2252**

### 🟢 Precision@3

**0.0667**

### 🟢 Precision@5

**0.0800**

### 🟡 Best Hybrid

**α = 0.75**

### 🟡 Cross-Encoder

**Evaluated but disabled by default**

### 🔵 Benchmark

**12 questions — 10 in-scope + 2 out-of-scope**

</div>

---

# 12. 🔬 Overall Conclusion

The retrieval experiments demonstrate that **chunk size, retrieval strategy, ranking, and retrieval depth all have a measurable impact on RAG retrieval quality**.

The **700–900 token configuration** provided the strongest overall semantic retrieval performance across the chunking experiments, achieving the highest MRR and strong evidence coverage.

Semantic Retrieval clearly outperformed BM25 in the tested benchmark.

The Hybrid experiments showed that **α = 0.75** was the strongest tested Hybrid configuration, providing a strong balance between semantic and lexical retrieval.

The Cross-Encoder increased Recall@5 and Recall@10, but reduced MRR. Therefore, it was not selected as the default reranking component.

The Precision@K analysis revealed that the retriever can find relevant evidence but also retrieves a substantial amount of non-relevant material. This is an important observation for the next stage because **retrieval quality alone is not sufficient; the generation layer must be explicitly constrained to use only supported evidence.**

Therefore, the validated retrieval layer for the next stage is:

> **700–900 token chunks + 10% overlap + 384-dimensional semantic embeddings + Semantic Retrieval + Top-10**

---

# 13. 🚀 Transition to the Next Stage

The retrieval layer is now validated and its artifacts have been saved.

The next stage will focus on **RAG Answer Generation and Clinical Safety**.

The next notebook will implement and evaluate:

1. **System Prompt / Safety Contract**
2. **Structured Clinical Answer Format**
3. **Evidence-grounded Citations**
4. **Claim-to-Evidence Traceability**
5. **Refusal Behavior**
6. **Confidence Levels**
7. **Groundedness and Citation Coverage**
8. **Out-of-Scope Safety Testing**

The generation layer will use the validated retrieval artifacts from this notebook.

<div style="background-color:#e3f2fd; border-left:6px solid #1565c0; padding:15px; border-radius:6px;">

# 🚀 NEXT STAGE

## Notebook 03 — RAG Generation & Clinical Safety

**Retrieval is validated.**

**Evidence artifacts are saved.**

**The next objective is to generate clinical answers that are strictly grounded in the retrieved guideline evidence, traceable through citations, and safe under out-of-scope or insufficient-evidence conditions.**

</div>

---

## ✅ NOTEBOOK 02 STATUS

<div style="background-color:#e8f5e9; border:2px solid #2e7d32; padding:20px; border-radius:8px;">

# 🟢 RETRIEVAL EVALUATION — COMPLETED

**Chunking evaluation ✓**

**Semantic Retrieval ✓**

**BM25 baseline ✓**

**Hybrid Retrieval ✓**

**Hybrid weight tuning ✓**

**Cross-Encoder evaluation ✓**

**Top-K evaluation ✓**

**Precision@K ✓**

**Benchmark validation ✓**

**Out-of-Scope validation ✓**

**Final configuration selected ✓**

**Retrieval artifacts saved ✓**

### 🏆 RETRIEVAL PHASE COMPLETE

</div>
